# W13｜影片的時間結構、編碼與容器

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/willismax/MediaSystem-Python-Course/blob/main/course-115-v2/colab/W13_%E5%BD%B1%E7%89%87%E7%B7%A8%E7%A2%BC%E8%88%87%E5%AE%B9%E5%99%A8.ipynb)

- **對應講義**：[HackMD 第 13 章　影片的時間結構、編碼與容器](https://hackmd.io/YHLQNX54SGey5O_O5c4g1g)
- **互動實驗室**：[W13 Lab（瀏覽器版）](https://willismax.github.io/MediaSystem-Python-Course/labs/w13/)
- **先備**：W10 取樣與頻譜、W11 pydub 剪輯、W12 Whisper（都用過 FFmpeg）；W02 影像是 `H × W × 3` 的陣列

W10–W12 處理的是**一條**隨時間變化的訊號（聲音）。影片則是「**一連串影像＋時間戳＋聲音**」：每秒 30 張 1080p 影像，不壓縮的話每秒約 1.49 Gbps。本週拆開一支影片的三層結構：**時間結構**（影格、影格率、GOP）、**編碼**（預測 → 轉換 → 量化 → 熵編碼、I／P／B 影格）、**容器**（MP4／MKV 只是「盒子」），並用 FFmpeg、ffprobe、PyAV、VMAF、PySceneDetect 做**只改一個變因**的控制實驗。

| 學習目標 | 你要能做到 |
|---|---|
| 時間結構 | 用「影格數 ≈ 時間 × 影格率」估算影格數；分辨 CFR／VFR（`r_frame_rate` vs `avg_frame_rate`）|
| 資料量 | 算出未壓縮資料率（1080p30 ≈ 1.49 Gbps）與「檔案大小 ≈ 總位元率 × 時間 ÷ 8」|
| 壓縮原理 | 說明空間／時間冗餘、量化為什麼會失真、`yuv420p` 為什麼省資料 |
| 動作預測與 GOP | 實作區塊比對找出 motion vector；用 PyAV 看出 I／P／B 影格與 GOP 結構 |
| Codec／Encoder／Container | 分辨三者；知道 `-c copy` 能做與不能做的事、`.mp4` 不保證是 H.264 |
| 控制實驗 | 一次只改一個變因（CRF、解析度、場景），用 ffprobe、檔案大小、編碼時間、PSNR／SSIM／VMAF 與截圖下結論 |

**使用方式**

- 從上到下依序執行即可。**CPU 執行階段就夠了**，不需要 GPU（FFmpeg 的軟體編碼器只用 CPU）；整本約需 4–6 分鐘，最久的是 13.25 的 CRF 實驗。
- 本週**不需要上傳任何檔案**：0.4 會用 FFmpeg 的 `lavfi` 虛擬來源，自動產生一支 1920×1080、30 fps、10 秒、含聲音的 `input.mp4`（講義的指令都用這個檔名）。想用自己的影片做練習，把 0.5 的 `UPLOAD` 改成 `True`。
- 需要網路的步驟：安裝套件、（必要時）安裝中文字型、（Colab 的 FFmpeg 沒有 VMAF 時）下載含 libvmaf 的靜態版 FFmpeg。任何一步失敗都會**印出原因並改用替代方案繼續**；最後的「執行摘要」會列出哪些步驟真的執行（real）、哪些用了替代方案（fallback）或略過（skipped）。
- 圖表的軸標籤與標題用英文（避免 Colab 缺中文字型時顯示成方塊）。

## 本週流程地圖

| 段落 | 主題 | 主要程式／指令 |
|---|---|---|
| 0 | 安裝、工具函式、字型、產生 `input.mp4`、（選用）上傳自己的影片 | `run()`、`probe()`、`grab()`、`quality()` |
| 13.1–13.3 | 影格、影格率、CFR vs VFR | `r_frame_rate`、`avg_frame_rate` |
| 13.4–13.6 | 解析度、未壓縮資料率、位元率與檔案大小 | `raw_rate()`、`file_size_MB()` |
| 13.8–13.9 | 空間／時間冗餘、預測 → 轉換 → 量化 → 熵編碼 | `scipy.fft.dctn`、`zlib` |
| 13.10 | YUV／YCbCr 與色度取樣 4:4:4／4:2:2／4:2:0 | `rgb_to_ycbcr()`、PyAV `frame.planes` |
| 13.11 | 動作估計：區塊比對、motion vector、殘差 | `block_match()` |
| 13.12–13.13 | I／P／B 影格與 GOP | PyAV `frame.pict_type`、`packet.size`；`ffprobe -show_entries frame=…` |
| 13.14 | 靜態 vs 複雜場景、場景偵測 | 每秒位元率、PySceneDetect `detect()` |
| 13.15 | Stream Copy vs 重新編碼 | `ffmpeg -ss … -c copy` |
| 13.16–13.17 | Codec／Encoder／Container、Mux／Demux | `-map`、`-c copy`、`-f md5` |
| 13.18 | H.264／H.265／AV1／VP9 比較 | `ffmpeg -encoders` |
| 13.20–13.21 | CBR／VBR／CRF、Preset | `-crf`、`-b:v`、`-preset` |
| 13.22–13.24 | ffprobe 與 Video Technical Card | `ffprobe -show_entries … -of json` |
| 13.25 | 控制實驗 A（CRF）、B（解析度）、C（場景）| `psnr`、`ssim`、`libvmaf` 濾鏡 |
| 工具 | 開源 GUI 工具：LosslessCut、MediaInfo、HandBrake；WebCodecs | |
| 練習 1–3、延伸 1–3 | Technical Card、資料量、CRF 實驗；場景、剪輯、Codec 比較 | |
| 檢核 | 執行摘要、檢核題、繳交清單 | |

## 0.1 安裝套件

| 套件 | 授權 | 用途 |
|---|---|---|
| FFmpeg（Colab 內建）| LGPL／GPL | 本週主角：轉檔、剪輯、濾鏡、品質量測；`ffprobe` 讀技術資訊 |
| [PyAV](https://github.com/PyAV-Org/PyAV) `av` | BSD-3-Clause | FFmpeg 函式庫（libavformat／libavcodec）的 Python 綁定：逐個 packet、逐張 frame 讀取，看 I／P／B 與每格大小 |
| [PySceneDetect](https://github.com/Breakthrough/PySceneDetect) `scenedetect` | BSD-3-Clause | 自動偵測鏡頭切換（scene cut），下週剪輯會用到 |
| scipy、pandas、matplotlib | BSD | DCT、表格、畫圖 |

> PyAV 的 pip 套件**自帶**一份 FFmpeg 函式庫，版本可能和 Colab 系統的 `ffmpeg` 指令不同；所以本 notebook 讀屬性時都會先檢查型別（見 13.12）。PySceneDetect 需要 OpenCV（Colab 已內建）：0.6.x 版要寫 `pip install scenedetect[opencv-headless]`，0.7 版起已直接相依 `opencv-python`，寫 `pip install scenedetect` 即可。

In [ ]:
!pip -q install av scenedetect scipy pandas matplotlib

## 0.2 匯入與工具函式

| 函式 | 用途 |
|---|---|
| `run(cmd)` | 執行 ffmpeg／ffprobe。`cmd` 可以直接貼**講義原樣的字串**；執行時會在 `ffmpeg` 後面自動補上 `-y -hide_banner -loglevel error`（覆寫舊檔、只顯示錯誤），**不影響編碼結果**。回傳（結果, 秒數）|
| `probe(path, entries)` | `ffprobe -of json`，回傳 Python dict |
| `info(path)` | 整理成一列：容器、codec、解析度、fps、時長、大小（MB）、位元率（Mbps）|
| `grab(path, t, size)` | 用 ffmpeg 取出某個時間點的一張影格，變成 NumPy 陣列 |
| `quality(dist, ref)` | 用 FFmpeg 內建 `psnr`、`ssim` 濾鏡比較「壓縮後」與「參考」影片 |
| `save_crops()` | 把幾張**原尺寸（1:1）**裁切圖並排存成 PNG（可再用最近鄰放大），用來比較壓縮瑕疵 |

單位約定：本 notebook 的 MB、GB、Mbps 都是**十進位**（1 MB = 10⁶ bytes、1 Mbps = 10⁶ bit/s），和講義、ffprobe 一致。

In [ ]:
import json
import os
import platform
import re
import shlex
import shutil
import subprocess
import sys
import time
import warnings
import zlib
from fractions import Fraction
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

warnings.filterwarnings("ignore", category=DeprecationWarning)
plt.rcParams.update({"figure.figsize": (11, 3.2), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.precision", 3)
IN_COLAB = "google.colab" in sys.modules
STATUS = {}          # 步驟 → "real"／"fallback"／"skipped"
TYPE_COLORS = {"I": "#d62728", "P": "#1f77b4", "B": "#2ca02c", "?": "#7f7f7f"}
QUIET = ["-y", "-hide_banner", "-loglevel", "error"]


def pkg_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "未安裝"


def run(cmd, quiet=True, check=True, show=False):
    '''執行 ffmpeg／ffprobe。cmd：講義原樣字串或 list。回傳 (CompletedProcess, 秒數)。'''
    args = shlex.split(cmd) if isinstance(cmd, str) else [str(c) for c in cmd]
    if quiet and Path(args[0]).name == "ffmpeg":
        args = [args[0], *QUIET, *args[1:]]
    if show:
        print("$", cmd if isinstance(cmd, str) else shlex.join(args))
    t0 = time.perf_counter()
    p = subprocess.run(args, capture_output=True, text=True, env={**os.environ, "SVT_LOG": "1"})
    sec = time.perf_counter() - t0
    if check and p.returncode != 0:
        raise RuntimeError(f"指令失敗（代碼 {p.returncode}）：{shlex.join(args)[:300]}\n{p.stderr[-1200:]}")
    return p, sec


def probe(path, entries=None, select=None):
    '''ffprobe → dict。entries 例如 "stream=codec_name,width:format=duration"。'''
    cmd = ["ffprobe", "-v", "error"]
    if select:
        cmd += ["-select_streams", select]
    cmd += ["-show_entries", entries] if entries else ["-show_format", "-show_streams"]
    p, _ = run(cmd + ["-of", "json", str(path)])
    return json.loads(p.stdout or "{}")


def to_float(x):
    try:
        v = float(x)
        return v if np.isfinite(v) else None
    except (TypeError, ValueError):
        return None


def fps_value(s):
    try:
        f = Fraction(str(s))
        return float(f) if f else None
    except (ValueError, ZeroDivisionError):
        return None


def info(path):
    d = probe(path)
    fmt = d.get("format", {})
    streams = d.get("streams", [])
    v = next((s for s in streams if s.get("codec_type") == "video"), {})
    a = next((s for s in streams if s.get("codec_type") == "audio"), {})
    size = int(fmt.get("size") or Path(path).stat().st_size)
    dur = to_float(fmt.get("duration")) or to_float(v.get("duration"))
    br = to_float(fmt.get("bit_rate")) or (size * 8 / dur if dur else None)
    return dict(file=Path(path).name, container=fmt.get("format_name"), vcodec=v.get("codec_name"),
                width=v.get("width"), height=v.get("height"), fps=fps_value(v.get("avg_frame_rate")),
                pix_fmt=v.get("pix_fmt"), acodec=a.get("codec_name"), duration_s=dur,
                size_MB=size / 1e6, bitrate_Mbps=br / 1e6 if br else None)


def video_size(path):
    s = probe(path, "stream=width,height", select="v:0")["streams"][0]
    return int(s["width"]), int(s["height"])


def grab(path, t=0.0, size=None, gray=False, extra_input_opts=()):
    '''取出時間 t（秒）的一張影格 → uint8 陣列 (H, W, 3) 或 (H, W)。size=(w, h) 會縮放。'''
    w, h = size or video_size(path)
    cmd = ["ffmpeg", *QUIET, *extra_input_opts, "-ss", f"{t:.4f}", "-i", str(path), "-frames:v", "1"]
    if size:
        cmd += ["-vf", f"scale={w}:{h}:flags=bicubic"]
    cmd += ["-f", "rawvideo", "-pix_fmt", "gray" if gray else "rgb24", "pipe:1"]
    raw = subprocess.run(cmd, capture_output=True, check=True).stdout
    shape = (h, w) if gray else (h, w, 3)
    return np.frombuffer(raw[: int(np.prod(shape))], np.uint8).reshape(shape)


def grab_index(path, n, fps=30, **kw):
    '''取第 n 張影格（從 0 起算）：把 -ss 設在影格中間之前，避免浮點誤差取錯格。'''
    return grab(path, max(0.0, (n - 0.5) / fps), **kw)


def psnr(a, b, peak=255.0):
    mse = np.mean((np.asarray(a, np.float64) - np.asarray(b, np.float64)) ** 2)
    return float("inf") if mse == 0 else 10 * np.log10(peak ** 2 / mse)


def align_graph(scale_to=None):
    '''把兩個輸入依「第幾張影格」重新編時間戳（settb + setpts=N），避免容器時間精度不同（例如 WebM／MKV 是 1 ms）
    造成 psnr／ssim／libvmaf 依時間戳配錯影格。scale_to=(w, h) 時先把壓縮後的影片放大到參考解析度。'''
    sc = f"scale={scale_to[0]}:{scale_to[1]}:flags=bicubic," if scale_to else ""
    return f"[0:v]{sc}settb=1/{FPS_ALIGN},setpts=N[d];[1:v]settb=1/{FPS_ALIGN},setpts=N[r];"


FPS_ALIGN = 30


def quality(dist, ref, metrics=("psnr", "ssim"), scale_to=None, ref_opts=(), psnr_log=None):
    '''FFmpeg 內建 psnr／ssim 濾鏡：第一個輸入是「壓縮後」，第二個是「參考」。
    scale_to=(w, h)：先把壓縮後的影片放大回參考解析度；ref_opts：參考影片的輸入選項（例如 -ss 4 -t 3）。'''
    out = {}
    for m in metrics:
        filt = m + (f"=stats_file={psnr_log}" if (m == "psnr" and psnr_log) else "")
        graph = f"{align_graph(scale_to)}[d][r]{filt}"
        cmd = ["ffmpeg", "-hide_banner", "-nostats", "-i", str(dist), *ref_opts, "-i", str(ref),
               "-lavfi", graph, "-f", "null", "-"]
        p, _ = run(cmd, quiet=False)
        if m == "psnr":
            hit = re.findall(r"PSNR .*?average:([\d.]+|inf)", p.stderr)
            out["PSNR_dB"] = float(hit[-1]) if hit else None
        if m == "ssim":
            hit = re.findall(r"SSIM .*?All:([\d.]+)", p.stderr)
            out["SSIM"] = float(hit[-1]) if hit else None
    return out


def save_crops(crops, labels, path, zoom=1, title=None):
    '''把同尺寸的裁切圖（原尺寸 1:1）並排：PNG 檔是像素精確的（最近鄰放大 zoom 倍），並在筆記本中顯示。'''
    gap = np.full((crops[0].shape[0], 6, 3), 255, np.uint8)
    strip = []
    for c in crops:
        c = c if c.ndim == 3 else np.repeat(c[..., None], 3, axis=2)
        strip += [c.astype(np.uint8), gap]
    montage = np.concatenate(strip[:-1], axis=1)
    montage = np.repeat(np.repeat(montage, zoom, axis=0), zoom, axis=1)
    plt.imsave(path, montage)
    fig, axes = plt.subplots(1, len(crops), figsize=(3.2 * len(crops), 3.2 * crops[0].shape[0] / crops[0].shape[1] + 0.6))
    for ax, c, lab in zip(np.atleast_1d(axes), crops, labels):
        ax.imshow(c, cmap="gray" if c.ndim == 2 else None, interpolation="nearest", vmin=0, vmax=255)
        ax.set_title(lab, fontsize=9)
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()
    print(f"已存檔 {path}（{montage.shape[1]}×{montage.shape[0]}，1:1 裁切 × {zoom} 倍最近鄰放大）")


def ffmpeg_names(kind, binary="ffmpeg"):
    '''解析 ffmpeg -encoders／-filters 的清單，回傳名稱集合（新版 FFmpeg 的濾鏡旗標欄只有 2 個字元）。'''
    p, _ = run([binary, "-hide_banner", f"-{kind}"], quiet=False, check=False)
    pat = r"^\s[VAS][A-Z.]{5}\s+(\S+)" if kind == "encoders" else r"^\s[A-Z.|]{2,3}\s+(\S+)\s+\S*->\S*"
    return {m.group(1) for line in p.stdout.splitlines() if (m := re.match(pat, line))}


p, _ = run(["ffmpeg", "-version"], quiet=False)
FFMPEG_VERSION = re.search(r"ffmpeg version (\S+)", p.stdout).group(1)
ENCODERS = ffmpeg_names("encoders")
FILTERS = ffmpeg_names("filters")
print(f"Python {platform.python_version()}｜FFmpeg {FFMPEG_VERSION}｜在 Colab：{IN_COLAB}")
print(f"numpy {np.__version__}｜pandas {pd.__version__}｜av {pkg_version('av')}｜scenedetect {pkg_version('scenedetect')}")
check = ["libx264", "libx265", "libsvtav1", "libaom-av1", "libvpx-vp9", "mpeg4", "ffv1", "aac"]
print("編碼器：", "  ".join(f"{e}{'✓' if e in ENCODERS else '✗'}" for e in check))
fcheck = ["drawtext", "psnr", "ssim", "libvmaf", "scale", "select", "concat"]
print("濾鏡：  ", "  ".join(f"{f}{'✓' if f in FILTERS else '✗'}" for f in fcheck))
assert "libx264" in ENCODERS, "這個 FFmpeg 沒有 libx264，本週大部分實驗無法進行"

## 0.3 字型（給 `drawtext` 用）

0.4 要用 FFmpeg 的 `drawtext` 濾鏡在畫面上「寫投影片文字」，它需要一個字型檔。Colab 預設**沒有中文字型**：

- `INSTALL_CJK_FONT = True`（預設）：找不到中文字型時，在 Colab 用 `apt-get install fonts-noto-cjk` 安裝 Noto Sans CJK（約 30–60 秒）。
- 安裝失敗或不在 Colab：改用 DejaVu Sans，投影片文字自動換成英文；後面所有實驗照常進行。

In [ ]:
INSTALL_CJK_FONT = True


def find_font():
    # 回傳 (字型檔路徑, 是否支援中文)
    if shutil.which("fc-list"):
        p = subprocess.run(["fc-list", ":lang=zh-tw", "file"], capture_output=True, text=True)
        files = sorted({ln.split(":")[0].strip() for ln in p.stdout.splitlines() if ln.strip()})
        for key in ["NotoSansCJK-Bold", "NotoSansCJK-Regular", "NotoSansCJK", "Sans"]:
            hit = [f for f in files if key in Path(f).name]
            if hit:
                return hit[0], True
        if files:
            return files[0], True
    for f in ["/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc",
              "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"]:
        if Path(f).exists():
            return f, True
    for f in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
              "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"]:
        if Path(f).exists():
            return f, False
    return None, False


FONT, FONT_CJK = find_font()
if not FONT_CJK and INSTALL_CJK_FONT and IN_COLAB:
    print("安裝 fonts-noto-cjk 中…")
    r = subprocess.run("apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1 || "
                       "(apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1)",
                       shell=True)
    FONT, FONT_CJK = find_font()

if FONT is None:
    STATUS["中文字型（drawtext）"] = "skipped"
    print("找不到任何字型：投影片將只有色塊、沒有文字（不影響後面的實驗）。")
else:
    STATUS["中文字型（drawtext）"] = "real" if FONT_CJK else "fallback"
    print(f"drawtext 字型：{FONT}｜中文：{'可' if FONT_CJK else '不可（改用英文文字）'}")

## 0.4 產生本週素材 `input.mp4`

用 FFmpeg 的 **lavfi 虛擬來源**（不需要任何檔案）合成一支 **1920×1080、30 fps、10 秒**的影片，接成三個場景：

| 時間 | 場景 | 來源 | 為什麼這樣設計 |
|---|---|---|---|
| 0–4 s | Static lecture（靜態投影片）| `color` 背景＋`drawbox` 色塊＋`drawtext` 文字＋一個緩慢移動的指標 `>` | 大面積平坦、相鄰影格幾乎一樣 → **好壓縮** |
| 4–7 s | High-motion detail（快速平移＋細節）| `testsrc2` 加上固定雜訊紋理，再用 `crop` 的時間運算式做快速平移 | 每個像素都在動、紋理細碎 → **難壓縮**；13.11 用來找 motion vector |
| 7–10 s | Cellular automaton（細胞自動機）| `cellauto`（規則 110）放大 | 畫面持續向上捲動、高頻細節多 → 第三個場景，給 PySceneDetect 偵測 |

聲音是 `sine`（每秒「嗶」一聲的 330 Hz 正弦波）→ AAC 128 kbps。

這支「母帶」本身以 `libx264 -crf 18` 高品質壓縮，並刻意設定 **`-g 60 -keyint_min 60 -sc_threshold 0`**：**每 2 秒一個關鍵影格**（0、2、4、6、8 s），很多手機、相機錄影也是每 1–2 秒一個；13.15 的剪輯實驗會用到這個特性。後面的品質量測都以 `input.mp4` 作為「參考」。

另外輸出一張**沒有經過 4:2:0 色度取樣**的投影片 `slide_444.png`（RGB），給 13.8–13.10 的 DCT 與色度取樣示範使用。

> 選用的來源（`color`、`testsrc2`、`cellauto`、`sine`）與濾鏡（`drawtext`、`drawbox`、`noise`、`crop`、`concat`、`colorchannelmixer`）在 FFmpeg 4.4（Colab）與 6.x 都有。

In [ ]:
W, H, FPS, DUR = 1920, 1080, 30, 10
SCENES = [("Static lecture", 0, 4), ("High-motion detail", 4, 7), ("Cellular automaton", 7, 10)]

if FONT_CJK:
    SLIDE_TEXT = ["第 13 週　影片的時間結構、編碼與容器", "• 影格數 ≈ 時間 × 影格率",
                  "• 未壓縮 1080p30 ≈ 1.49 Gbps", "• Codec ≠ Container", "4:2:0 色度取樣：紅字藍底的邊緣"]
else:
    SLIDE_TEXT = ["Week 13  Video: Time, Coding & Containers", "• frames ≈ time × frame rate",
                  "• raw 1080p30 ≈ 1.49 Gbps", "• Codec ≠ Container", "4:2:0 chroma: red text on blue"]
for i, s in enumerate(SLIDE_TEXT):
    Path(f"slide_t{i}.txt").write_text(s, encoding="utf-8")


def slide_chain():
    # 投影片場景的濾鏡鏈（RGB 中繪製，最後才轉 yuv420p）
    parts = ["format=rgb24",
             "drawbox=x=0:y=0:w=iw:h=150:color=0x1F3A5F:t=fill",
             "drawbox=x=120:y=300:w=1680:h=4:color=0xC8102E:t=fill",
             "drawbox=x=0:y=880:w=iw:h=140:color=0x1565C0:t=fill"]
    if FONT:
        spec = [(0, 64, "white", 120, 40), (1, 56, "0x222222", 160, 400), (2, 56, "0xC8102E", 160, 520),
                (3, 56, "0x1565C0", 160, 640), (4, 60, "0xFF3030", 160, 905)]
        parts += [f"drawtext=fontfile={FONT}:textfile=slide_t{i}.txt:fontsize={fs}:fontcolor={c}:x={x}:y={y}"
                  for i, fs, c, x, y in spec]
        parts.append(f"drawtext=fontfile={FONT}:text='>':fontsize=60:fontcolor=0xE65100:x=100+40*t:y=780")
    else:
        parts.append("drawbox=x='100+40*t':y=780:w=40:h=40:color=0xE65100:t=fill")
    return ",".join(parts)


def gen_cmd():
    return [
        "ffmpeg",
        "-t", "4", "-f", "lavfi", "-i", f"color=c=0xF4F1EA:s={W}x{H}:r={FPS}",
        "-t", "3", "-f", "lavfi", "-i", f"testsrc2=s=2880x1620:r={FPS}",
        "-t", "3", "-f", "lavfi", "-i", f"cellauto=s=480x270:r={FPS}:rule=110:random_fill_ratio=0.5:scroll=1",
        "-t", str(DUR), "-f", "lavfi", "-i", "sine=frequency=330:beep_factor=4:sample_rate=48000",
        "-filter_complex",
        f"[0:v]{slide_chain()},format=yuv420p,setsar=1[a];"
        "[1:v]noise=alls=10,crop=1920:1080:x='480+420*sin(1.6*t)':y='270+250*cos(1.1*t)',format=yuv420p,setsar=1[b];"
        "[2:v]scale=1920:1080:flags=neighbor,colorchannelmixer=rr=0.2:gg=0.6:bb=1.0,format=yuv420p,setsar=1[c];"
        "[a][b][c]concat=n=3:v=1:a=0[v];[3:a]volume=0.3,aformat=channel_layouts=stereo[aud]",
        "-map", "[v]", "-map", "[aud]",
        "-c:v", "libx264", "-crf", "18", "-preset", "veryfast", "-g", "60", "-keyint_min", "60", "-sc_threshold", "0",
        "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart", "input.mp4",
    ]


def make_slide_png():
    run(["ffmpeg", "-f", "lavfi", "-i", f"color=c=0xF4F1EA:s={W}x{H}:r={FPS}", "-vf", slide_chain(),
         "-frames:v", "1", "slide_444.png"])


try:
    _, sec = run(gen_cmd())
    make_slide_png()
    STATUS["產生素材 input.mp4"] = "real"
except RuntimeError as err:
    print("含文字的版本失敗（多半是字型問題），改用不含 drawtext 的簡化版：", str(err)[-300:])
    FONT = None
    _, sec = run(gen_cmd())
    make_slide_png()
    STATUS["產生素材 input.mp4"] = "fallback"
print(f"input.mp4 產生完成，耗時 {sec:.1f} s")

display(pd.DataFrame([info("input.mp4")]))
fig, axes = plt.subplots(1, 3, figsize=(13, 2.7))
for ax, (name, t0, t1) in zip(axes, SCENES):
    ax.imshow(grab("input.mp4", t0 + 1.0, size=(480, 270)))
    ax.set_title(f"{name} ({t0}-{t1} s)", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 0.5 選用：上傳自己的影片

練習 1–3 建議用**你自己用手機拍的影片**（10–60 秒即可）。把 `UPLOAD` 改成 `True` 會跳出上傳視窗；或把影片放進 Colab 左側檔案面板後，在 `MY_VIDEO_PATH` 填入檔名。兩者預設都關閉，所以「全部執行」不會停下來等你；沒有自己的影片時，練習會改用 `input.mp4` 示範。

> **隱私提醒**：手機影片的中繼資料可能含有**拍攝時間、裝置型號、甚至 GPS 位置**（ffprobe 的 `tags` 會看到）。繳交前請確認，或只繳交 Technical Card 而不繳交影片本身。

In [ ]:
UPLOAD = False        # 改成 True：上傳自己的影片
MY_VIDEO_PATH = ""    # 或填入已在 Colab 檔案面板中的檔名，例如 "my_phone.mp4"

MY_VIDEO = Path(MY_VIDEO_PATH) if MY_VIDEO_PATH and Path(MY_VIDEO_PATH).exists() else None
if UPLOAD and MY_VIDEO is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            MY_VIDEO = Path(next(iter(uploaded)))
    except Exception as err:
        print(f"上傳失敗（{type(err).__name__}: {err}）。不在 Colab 時請改用 MY_VIDEO_PATH。")

if MY_VIDEO:
    STATUS["自己的影片 MY_VIDEO"] = "real"
    display(pd.DataFrame([info(MY_VIDEO)]))
else:
    STATUS["自己的影片 MY_VIDEO"] = "skipped"
    print("沒有自己的影片：練習 1–3 會改用 input.mp4 示範（繳交時請換成自己的影片）。")

## 13.1–13.3 影格、影格率、CFR 與 VFR

**概念**：影片 = 一張一張的**影格（frame）**，每張帶有一個**時間戳（timestamp）**。

  影格數 ≈ 時間 T（秒）× 影格率 f（fps）

例如 10 秒 × 30 fps = 300 張；60 秒 × 30 fps = 1800 張。寫成「≈」是因為實際影片的開頭、結尾、掉格與 VFR 都會讓數字差一點點。

**13.2 常見影格率**

| fps | 常見用途 | 備註 |
|---|---|---|
| 23.976／24 | 電影、劇情片 | 電影感；24000/1001 ≈ 23.976 |
| 25 | 歐洲等 PAL 地區電視 | |
| 29.97／30 | 臺灣、美國等 NTSC 地區電視、網路影片、手機預設 | 30000/1001 ≈ 29.97 |
| 50／60 | 運動、遊戲、螢幕錄影 | 動作更順，但資料量更多 |
| 120／240 | 慢動作 | 播放時用 30 fps 放，就慢 4／8 倍 |

**13.3 CFR vs VFR**：固定影格率（Constant Frame Rate）每格間隔相同；**手機錄影、螢幕錄影常是 VFR**（Variable Frame Rate），畫面不動或光線不足時影格間隔會變。ffprobe 有兩個欄位：

- `r_frame_rate`：串流的「基本」影格率（能表示所有時間戳的最小時間單位推得的值，**不是平均值**）。
- `avg_frame_rate`：總影格數 ÷ 總時長，**平均**影格率。

兩者不一樣 → 很可能是 VFR。VFR 素材直接丟進剪輯軟體可能造成**聲音和畫面不同步**，常見做法是先轉成 CFR。

> **先預測再執行**：`input.mp4` 有幾張影格？如果把一段 30 fps 的影片「每 3 格只留 1 格，只有中間 1 秒全部保留」，`r_frame_rate` 和 `avg_frame_rate` 各會是多少？

In [ ]:
def frame_count(seconds, fps):
    return round(seconds * fps)


assert frame_count(10, 30) == 300
assert frame_count(60, 30) == 1800

s = probe("input.mp4", "stream=nb_frames,r_frame_rate,avg_frame_rate,duration", select="v:0")["streams"][0]
p, _ = run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_packets",
            "-show_entries", "stream=nb_read_packets", "-of", "csv=p=0", "input.mp4"])
print(f"預測：{DUR} s × {FPS} fps = {frame_count(DUR, FPS)} 張")
print(f"ffprobe：nb_frames = {s.get('nb_frames')}，實際數到的 packet = {p.stdout.strip()}，"
      f"r_frame_rate = {s.get('r_frame_rate')}，avg_frame_rate = {s.get('avg_frame_rate')}")

fps_table = pd.DataFrame({"fps": [24000 / 1001, 24, 25, 30000 / 1001, 30, 50, 60, 120]})
fps_table["frame_interval_ms"] = 1000 / fps_table.fps
fps_table["frames_in_60s"] = (fps_table.fps * 60).round().astype(int)
display(fps_table)

In [ ]:
# 13.3 做一支 VFR 影片：前 3 秒，「每 3 格留 1 格」，但 1–2 秒之間全部保留
vfr_filter = "scale=640:360,select='not(mod(n\\,3))+between(t\\,1\\,2)'"
for sync in (["-fps_mode", "vfr"], ["-vsync", "vfr"]):          # FFmpeg 5.1+ 用 -fps_mode；4.4 用 -vsync
    p, _ = run(["ffmpeg", "-t", "3", "-i", "input.mp4", "-an", "-vf", vfr_filter, *sync,
                "-c:v", "libx264", "-crf", "23", "vfr.mp4"], check=False)
    if p.returncode == 0:
        break
run(["ffmpeg", "-t", "3", "-i", "input.mp4", "-an", "-vf", "scale=640:360", "-c:v", "libx264", "-crf", "23", "cfr.mp4"])

rows, deltas = [], {}
for f in ["cfr.mp4", "vfr.mp4"]:
    s = probe(f, "stream=r_frame_rate,avg_frame_rate,nb_frames", select="v:0")["streams"][0]
    rows.append(dict(file=f, r_frame_rate=s["r_frame_rate"], avg_frame_rate=s["avg_frame_rate"],
                     avg_fps=round(fps_value(s["avg_frame_rate"]), 2), nb_frames=s.get("nb_frames")))
    ts = probe(f, "packet=pts_time", select="v:0")["packets"]
    t = np.sort([to_float(x["pts_time"]) for x in ts if to_float(x.get("pts_time")) is not None])
    deltas[f] = (t[1:], np.diff(t) * 1000)
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(11, 2.8))
for f, (t, d) in deltas.items():
    ax.step(t, d, where="post", label=f)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frame interval (ms)")
ax.set_title("CFR vs VFR: time between consecutive frames")
ax.legend()
plt.show()

**觀察**

- `input.mp4` 的 300 張與 10 × 30 完全吻合（CFR，`r_frame_rate` = `avg_frame_rate` = 30/1）。
- `vfr.mp4` 的 `r_frame_rate` 仍是 30/1，但 `avg_frame_rate` 變成一個奇怪的分數（約 17 fps）；圖上影格間隔在 33 ms 與 100 ms 之間跳動。**只看 `r_frame_rate` 會誤以為它是 30 fps 的 CFR**。
- 練習 1 用自己的手機影片時，記得比較這兩個欄位。

## 13.4–13.6 解析度、未壓縮資料率、位元率與檔案大小

**13.4 解析度**：1920 × 1080 = 2,073,600 像素；3840 × 2160（4K UHD）= 8,294,400 像素，是 1080p 的 **4 倍**（寬、高各 2 倍）。

**13.5 未壓縮資料率**（RGB、每個顏色 8 bit = 每像素 3 bytes）：

  1920 × 1080 × 3 × 30 = 186,624,000 B/s ≈ 186.6 MB/s ≈ 1493 Mbps ≈ **1.49 Gbps**；一分鐘 ≈ **11.2 GB**

**13.6 位元率與檔案大小**：

  File Size（MB）≈ Total Bitrate（Mbps）× Duration（s）÷ 8

例如影像＋聲音總位元率 8.128 Mbps、60 秒：8.128 × 60 ÷ 8 ≈ **60.96 MB**。÷ 8 是因為 1 byte = 8 bit。

> **先預測再執行**：4K 的像素是 1080p 的 4 倍，用同樣的編碼設定壓縮後，**檔案也會剛好大 4 倍嗎**？`input.mp4` 比未壓縮小了幾倍？

In [ ]:
def pixels(w, h):
    return w * h


def raw_rate(w, h, fps, bytes_per_pixel=3):
    # 未壓縮資料率（bytes/s）
    return w * h * bytes_per_pixel * fps


def file_size_MB(total_mbps, seconds):
    # File Size ≈ Total Bitrate × Duration / 8
    return total_mbps * seconds / 8


assert pixels(1920, 1080) == 2_073_600
assert pixels(3840, 2160) == 8_294_400
assert pixels(3840, 2160) / pixels(1920, 1080) == 4

r = raw_rate(1920, 1080, 30)
assert r == 186_624_000
assert round(r / 1e6, 1) == 186.6                  # MB/s
assert round(r * 8 / 1e6) == 1493                  # Mbps
assert round(r * 8 / 1e9, 2) == 1.49               # Gbps
assert round(r * 60 / 1e9, 1) == 11.2              # GB / 分鐘
assert abs(file_size_MB(8.128, 60) - 60.96) < 1e-9

print(f"1080p30 RGB 未壓縮：{r:,} B/s = {r / 1e6:.1f} MB/s = {r * 8 / 1e6:.0f} Mbps = {r * 8 / 1e9:.2f} Gbps；"
      f"一分鐘 {r * 60 / 1e9:.1f} GB")
print(f"8.128 Mbps × 60 s ÷ 8 = {file_size_MB(8.128, 60):.2f} MB")

inp = info("input.mp4")
raw_10s = raw_rate(W, H, FPS) * inp["duration_s"]
print(f"\ninput.mp4：總位元率 {inp['bitrate_Mbps']:.2f} Mbps × {inp['duration_s']:.1f} s ÷ 8 = "
      f"{file_size_MB(inp['bitrate_Mbps'], inp['duration_s']):.2f} MB（實際 {inp['size_MB']:.2f} MB）")
print(f"同長度的未壓縮 RGB：{raw_10s / 1e9:.2f} GB → 壓縮比約 {raw_10s / (inp['size_MB'] * 1e6):.0f} : 1")

**觀察**

- 公式和 ffprobe 的位元率算出來的大小幾乎一樣（差一點點來自容器的額外資料，例如索引表）。
- 壓縮比上百倍，代表「影片檔」裡存的**絕對不是**一張張完整的影像。接下來三節（13.8–13.11）就是在拆解這上百倍是怎麼來的。
- **位元率 ≠ 品質**：同樣 5 Mbps，給靜態投影片綽綽有餘，給快速運動的畫面可能滿是馬賽克（13.14、13.25 實驗 C）。比較品質要看**內容＋位元率＋編碼器**，不能只看位元率。

## 13.8–13.9 壓縮原理：冗餘、轉換、量化

**概念**：影片能壓縮，是因為裡面有大量**冗餘（redundancy）**：

| 冗餘 | 意思 | 編碼器怎麼利用 |
|---|---|---|
| 空間冗餘（spatial）| 同一張影格中，相鄰像素很像（天空、牆壁、投影片背景）| **畫面內（intra）預測**＋轉換 |
| 時間冗餘（temporal）| 相鄰影格很像（背景不動、物體只移動一點）| **畫面間（inter）預測**：motion vector＋殘差（13.11）|
| 感知冗餘 | 人眼對亮度細節敏感、對顏色細節遲鈍 | 色度取樣 4:2:0（13.10）、量化 |

**畫面內（intra）編碼管線**：

```
原始區塊 ──► Prediction ──► Transform ──► Quantization ──► Entropy Coding ──► 位元串流
            用鄰近已解碼     DCT：把像素     把係數除以步階再     把「很多 0、少數小數字」
            像素猜這一塊     變成頻率係數    四捨五入（唯一      用更短的碼表示
            → 只留殘差       （可逆）        真正丟資訊的步驟）  （可逆）
```

**量化範例**：一列頻率係數 `13 4 2 1 0.5`，低頻用小步階、高頻用大步階（步階 `1 1 5 5 5`）：

  round(13/1)·1 = 13、round(4/1)·1 = 4、round(2/5)·5 = 0、round(1/5)·5 = 0、round(0.5/5)·5 = 0 → `13 4 0 0 0`

那三個變成 0 的數字**再也回不來**——這就是有損壓縮「損」在哪裡。但一長串 0 非常好壓（熵編碼）。

In [ ]:
coeffs = np.array([13, 4, 2, 1, 0.5])
steps = np.array([1, 1, 5, 5, 5])              # 低頻步階小、高頻步階大
levels = np.round(coeffs / steps)              # 量化：實際存進檔案的整數
recon = levels * steps                         # 反量化：解碼端只能還原到這裡
print("原始係數：", coeffs, "\n量化後整數：", levels.astype(int), "\n反量化：", recon)
assert recon.tolist() == [13, 4, 0, 0, 0]

### 動手做：8×8 區塊 DCT → 量化 → 「熵編碼」

用 `slide_444.png` 的亮度（Y）做和 JPEG／H.264 intra 類似的流程：

1. 切成 8×8 區塊，每塊減 128 後做 2D DCT（`scipy.fft.dctn(..., norm="ortho")`，正交 → 可完全還原）。
2. 用 JPEG 標準亮度量化表 × `scale` 量化（`scale` 越大＝越粗＝越像高 CRF）。
3. 數一數有多少係數變成 0；反量化 → 反 DCT → 和原圖算 **PSNR**（Peak Signal-to-Noise Ratio，越高越像，單位 dB）。
4. 用 `zlib` 壓縮量化後的整數，當作「熵編碼」的替身（真正的 H.264 用 CAVLC／CABAC，效果更好）。

> **先預測再執行**：`scale` 從 1 調到 8，零係數比例、PSNR、壓縮後大小各會怎麼變？瑕疵會先出現在平坦背景還是文字邊緣？

In [ ]:
from scipy.fft import dctn, idctn

JPEG_Q = np.array([
    [16, 11, 10, 16, 24, 40, 51, 61], [12, 12, 14, 19, 26, 58, 60, 55],
    [14, 13, 16, 24, 40, 57, 69, 56], [14, 17, 22, 29, 51, 87, 80, 62],
    [18, 22, 37, 56, 68, 109, 103, 77], [24, 35, 55, 64, 81, 104, 113, 92],
    [49, 64, 78, 87, 103, 121, 120, 101], [72, 92, 95, 98, 112, 100, 103, 99]], float)

slide_rgb = (plt.imread("slide_444.png")[..., :3] * 255).round().astype(np.uint8)
Y = slide_rgb @ np.array([0.299, 0.587, 0.114])          # BT.601 亮度
Hb, Wb = Y.shape[0] // 8, Y.shape[1] // 8


def to_blocks(img):
    return img[:Hb * 8, :Wb * 8].reshape(Hb, 8, Wb, 8).transpose(0, 2, 1, 3)


def from_blocks(b):
    return b.transpose(0, 2, 1, 3).reshape(Hb * 8, Wb * 8)


C = dctn(to_blocks(Y) - 128, axes=(2, 3), norm="ortho")
assert np.allclose(from_blocks(idctn(C, axes=(2, 3), norm="ortho")) + 128, Y[:Hb * 8, :Wb * 8])  # DCT 本身可逆

raw_bytes = Y.size                                         # 8-bit 灰階未壓縮
lossless = len(zlib.compress(np.round(Y).astype(np.uint8).tobytes(), 9))
rows, recons = [], {}
for scale in [0.5, 1, 3, 8]:
    q = np.round(C / (JPEG_Q * scale))                     # 量化
    rec = from_blocks(idctn(q * JPEG_Q * scale, axes=(2, 3), norm="ortho")) + 128
    rec = np.clip(rec, 0, 255)
    recons[scale] = rec
    # 同一個頻率的係數放在一起（類似 zig-zag 掃描後的長串 0），再用 zlib 壓縮
    packed = q.astype(np.int16).transpose(2, 3, 0, 1).tobytes()
    rows.append(dict(scale=scale, zero_coeffs=f"{np.mean(q == 0):.1%}",
                     PSNR_dB=round(psnr(Y[:Hb * 8, :Wb * 8], rec), 2),
                     zlib_bytes=len(zlib.compress(packed, 9)),
                     vs_raw=f"{raw_bytes / len(zlib.compress(packed, 9)):.0f} : 1"))
print(f"亮度未壓縮 {raw_bytes:,} bytes；直接 zlib（無損）{lossless:,} bytes")
dct_df = pd.DataFrame(rows)
display(dct_df)

# 一個跨過文字邊緣的 8×8 區塊：量化前後的係數
by, bx = 400 // 8 + 2, 160 // 8 + 3
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
axes[0].imshow(to_blocks(Y)[by, bx], cmap="gray", vmin=0, vmax=255)
axes[0].set_title("8x8 block (pixels)")
for ax, (lab, mat) in zip(axes[1:], [("DCT coefficients", C[by, bx]),
                                       ("Quantized levels (scale=3)", np.round(C[by, bx] / (JPEG_Q * 3)))]):
    im = ax.imshow(np.abs(mat), cmap="magma")
    ax.set_title(lab)
    fig.colorbar(im, ax=ax)
for ax in axes:
    ax.grid(False)
plt.tight_layout()
plt.show()

y0, x0 = 392, 150
crops = [np.round(Y[y0:y0 + 64, x0:x0 + 160]).astype(np.uint8)] + \
        [np.round(recons[s][y0:y0 + 64, x0:x0 + 160]).astype(np.uint8) for s in [1, 3, 8]]
save_crops(crops, ["original Y", "scale=1", "scale=3", "scale=8"], "dct_crops.png", zoom=3,
           title="8x8 DCT + quantization: 1:1 crops (blocking & ringing near edges)")

**觀察**

- 零係數比例隨 `scale` 上升，zlib 後的大小快速下降，PSNR 也跟著下降——**「位元數 vs 品質」的取捨就在量化這一步**。
- 量化前的係數集中在左上角（低頻）；量化後大多變成 0，只剩少數幾個整數。
- 注意：這張投影片大部分是純色，**直接用 zlib 無損壓縮**反而比上表都小——因為 zlib 處理 int16 係數只是很粗糙的「熵編碼替身」。真正的編碼器還有畫面內預測、zig-zag 掃描、run-length 與 CABAC 等專門設計，而 DCT＋量化的優勢在照片、影片這類**自然影像**上最明顯（無損壓縮對它們通常只能做到約 2:1）。
- 1:1 裁切放大後，`scale=8` 在文字邊緣出現**振鈴（ringing）**與 8×8 的**方塊（blocking）**；平坦背景幾乎看不出差別——因為平坦區塊本來就只有 DC 一個係數。
- H.264 的轉換是 4×4／8×8 的整數 DCT 近似、H.265 可到 32×32，並在解碼端加上**去方塊濾波（deblocking filter）**淡化區塊邊界。

## 13.10 YUV／YCbCr 與色度取樣

**概念**：影片幾乎都不用 RGB 存，而是把顏色拆成：

- **Y**（luma，亮度）：Y = 0.299 R + 0.587 G + 0.114 B（BT.601；HD 影片常用 BT.709 係數，概念相同）
- **Cb、Cr**（chroma，色差）：藍色差、紅色差

人眼對**亮度細節**敏感、對**顏色細節**遲鈍，所以可以只降低 Cb／Cr 的解析度：

| 取樣 | Cb／Cr 解析度（以 1920×1080 為例）| bytes／像素（8 bit）| 常見用途 |
|---|---|---|---|
| 4:4:4 | 1920×1080（完整）| 3 | 螢幕錄影、電腦圖形、後製中間檔 |
| 4:2:2 | 960×1080（水平減半）| 2 | 專業攝影機、廣播 |
| 4:2:0 | 960×540（水平、垂直都減半）| **1.5** | 手機、串流、網路影片（幾乎全部）|

**`yuv420p` 是像素格式（pixel format）**：Y、U（Cb）、V（Cr）三個平面（planar, p），色度 4:2:0。它**不是解析度、也不是 codec**——一支 H.264 影片可以是 yuv420p 或 yuv444p；yuv420p 也可以裝在 H.265、AV1 裡。很多播放器、瀏覽器只支援 yuv420p，所以轉檔時常加 `-pix_fmt yuv420p` 確保相容。

> **先預測再執行**：4:2:0 會讓**黑白文字**還是**紅字藍底**比較容易出現瑕疵？如果反過來把**亮度**降成一半解析度，會怎樣？

In [ ]:
def rgb_to_ycbcr(rgb):
    # BT.601 全範圍（JPEG）YCbCr；影片通常是 16–235 的「有限範圍」，概念相同
    r, g, b = [rgb[..., i].astype(np.float64) for i in range(3)]
    y = 0.299 * r + 0.587 * g + 0.114 * b
    cb = 128 - 0.168736 * r - 0.331264 * g + 0.5 * b
    cr = 128 + 0.5 * r - 0.418688 * g - 0.081312 * b
    return y, cb, cr


def ycbcr_to_rgb(y, cb, cr):
    r = y + 1.402 * (cr - 128)
    g = y - 0.344136 * (cb - 128) - 0.714136 * (cr - 128)
    b = y + 1.772 * (cb - 128)
    return np.clip(np.stack([r, g, b], -1), 0, 255).round().astype(np.uint8)


def subsample(c, mode):
    # 回傳 (縮小後的色度平面, 還原到全尺寸給顯示用)
    h, w = c.shape
    if mode == "4:4:4":
        small = c
    elif mode == "4:2:2":
        small = c.reshape(h, w // 2, 2).mean(axis=2)
    elif mode == "4:2:0":
        small = c.reshape(h // 2, 2, w // 2, 2).mean(axis=(1, 3))
    up = np.repeat(np.repeat(small, h // small.shape[0], axis=0), w // small.shape[1], axis=1)
    return small, up


y, cb, cr = rgb_to_ycbcr(slide_rgb)
fig, axes = plt.subplots(1, 4, figsize=(14, 2.4))
for ax, img, lab in zip(axes, [slide_rgb, y, cb, cr], ["RGB", "Y (luma)", "Cb (blue-difference)", "Cr (red-difference)"]):
    ax.imshow(img, cmap=None if img.ndim == 3 else "gray", vmin=0, vmax=255)
    ax.set_title(lab)
    ax.axis("off")
plt.tight_layout()
plt.show()

rows, versions = [], {}
for mode in ["4:4:4", "4:2:2", "4:2:0"]:
    cb_s, cb_up = subsample(cb, mode)
    cr_s, cr_up = subsample(cr, mode)
    nbytes = y.size + cb_s.size + cr_s.size                 # 每個樣本 1 byte
    versions[mode] = ycbcr_to_rgb(y, cb_up, cr_up)
    rows.append(dict(mode=mode, chroma_plane=f"{cb_s.shape[1]}x{cb_s.shape[0]}",
                     bytes_per_pixel=nbytes / y.size, frame_MB=nbytes / 1e6,
                     PSNR_RGB_dB=round(psnr(slide_rgb, versions[mode]), 2)))
# 錯誤示範：對「亮度」做 2×2 降取樣，色度保持完整
_, y_up = subsample(y, "4:2:0")
versions["luma 2x2 (wrong)"] = ycbcr_to_rgb(y_up, cb, cr)
rows.append(dict(mode="luma 2x2 (wrong)", chroma_plane="1920x1080", bytes_per_pixel=(y.size / 4 + 2 * y.size) / y.size,
                 frame_MB=(y.size / 4 + 2 * y.size) / 1e6, PSNR_RGB_dB=round(psnr(slide_rgb, versions["luma 2x2 (wrong)"]), 2)))
chroma_df = pd.DataFrame(rows)
display(chroma_df)
assert [r["bytes_per_pixel"] for r in rows[:3]] == [3.0, 2.0, 1.5]

y0, x0 = 896, 150
save_crops([v[y0:y0 + 64, x0:x0 + 128] for v in versions.values()], list(versions), "chroma_crops.png", zoom=4,
           title="Chroma subsampling on red text over blue: 1:1 crops")

再用 **PyAV** 直接看 `input.mp4` 解碼出來的一張影格：yuv420p 的三個平面大小。

In [ ]:
try:
    import av
    with av.open("input.mp4") as c:
        frame = next(c.decode(video=0))
        print(f"frame.format.name = {frame.format.name}，frame 寬×高 = {frame.width}×{frame.height}")
        for i, plane in enumerate(frame.planes):
            print(f"  plane {i}（{'YUV'[i]}）：{plane.width}×{plane.height}，line_size={plane.line_size}")
        n = sum(pl.width * pl.height for pl in frame.planes)
        print(f"一張影格的樣本數 = {n:,} = {n / (frame.width * frame.height)} bytes/像素（8 bit）")
    STATUS["PyAV 讀取 yuv420p 平面"] = "real"
except Exception as err:
    STATUS["PyAV 讀取 yuv420p 平面"] = "fallback"
    print(f"PyAV 無法使用（{type(err).__name__}: {err}）；改用 ffprobe：",
          probe("input.mp4", "stream=pix_fmt,width,height", select="v:0")["streams"][0])

**觀察**

- Y 看起來就是一張黑白照片，所有的「形狀」都在 Y；Cb、Cr 很平、細節很少。
- 4:2:0 每像素只要 1.5 bytes（省一半），PSNR 只掉一點；放大紅字藍底的邊緣，顏色會「溢出」一兩個像素（color bleeding）。如果反過來降低**亮度**解析度（色度保持完整），用了**更多**資料（2.25 bytes/像素），文字反而明顯變糊、PSNR 更低——這就是為什麼大家選擇犧牲色度。
- PyAV 顯示 `input.mp4` 解析度仍是 1920×1080，但 U、V 平面只有 960×540：**yuv420p 是像素格式，不是解析度**。
- 螢幕錄影的彩色細字（程式碼語法上色、Excel 的紅字）是 4:2:0 最容易出問題的內容；這時可以考慮 4:4:4（但相容性較差）。

## 13.11 動作估計：區塊比對、motion vector 與殘差

**概念**：相鄰兩張影格大部分內容一樣，只是「移動了一點」。編碼器把目前影格切成區塊（H.264 的巨集區塊是 16×16），對每一塊在**前一張影格**的附近找最像的位置：

- **Motion vector（MV）**：「這一塊是從前一張的哪裡搬過來的」（dx, dy）。
- **SAD**（Sum of Absolute Differences）：兩個區塊每個像素差的絕對值總和，越小越像。
- **殘差（residual）**：目前區塊 − 搬過來的預測區塊。編碼器只存 **MV＋殘差**（殘差再走 13.9 的轉換、量化、熵編碼）。

下面用 NumPy 實作最直接的**全搜尋（full search）**：16×16 區塊、搜尋範圍 ±8 像素。為了讓 CPU 快一點，先把影格縮成 960×540（MV 也跟著縮成一半）。

> **先預測再執行**：高動態場景是整個畫面在平移，MV 場應該長什麼樣子？有了動作補償，殘差能量會降多少？

In [ ]:
def block_match(prev, cur, B=16, R=8):
    # 全搜尋區塊比對：回傳每個區塊的 MV (dy, dx)、最小 SAD、動作補償後的預測影格
    nby, nbx = cur.shape[0] // B, cur.shape[1] // B
    cur = cur[:nby * B, :nbx * B].astype(np.float32)
    pad = np.pad(prev[:nby * B, :nbx * B].astype(np.float32), R, mode="edge")
    best = np.full((nby, nbx), np.inf, np.float32)
    mv = np.zeros((nby, nbx, 2), int)
    cands = sorted(((dy, dx) for dy in range(-R, R + 1) for dx in range(-R, R + 1)),
                   key=lambda d: abs(d[0]) + abs(d[1]))     # (0,0) 先試：SAD 相同時偏好「沒移動」
    for dy, dx in cands:
        ref = pad[R + dy:R + dy + nby * B, R + dx:R + dx + nbx * B]
        sad = np.abs(cur - ref).reshape(nby, B, nbx, B).sum(axis=(1, 3))
        better = sad < best
        best[better] = sad[better]
        mv[better] = (dy, dx)
    pred = np.empty_like(cur)
    for i in range(nby):
        for j in range(nbx):
            dy, dx = mv[i, j]
            pred[i * B:(i + 1) * B, j * B:(j + 1) * B] = pad[R + i * B + dy:R + (i + 1) * B + dy,
                                                             R + j * B + dx:R + (j + 1) * B + dx]
    return mv, best, pred, cur


N0 = 138                                   # 第 138 張 = 4.6 s（高動態場景開始後 0.6 s）
SMALL = (960, 540)
f0 = grab_index("input.mp4", N0, size=SMALL, gray=True)
f1 = grab_index("input.mp4", N0 + 1, size=SMALL, gray=True)
t_start = time.perf_counter()
mv, sad, pred, cur = block_match(f0, f1)
print(f"區塊比對：{mv.shape[0]}×{mv.shape[1]} 個區塊 × 289 個候選位置，耗時 {time.perf_counter() - t_start:.2f} s")

prev_crop = f0[:cur.shape[0], :cur.shape[1]].astype(np.float32)
e_no_mc = float(np.sum((cur - prev_crop) ** 2))
e_mc = float(np.sum((cur - pred) ** 2))
print(f"殘差能量（平方和）：不做動作補償 {e_no_mc:.3e}｜做動作補償 {e_mc:.3e}｜降為 {e_mc / e_no_mc:.1%}")

# 理論值：crop 的 x = 480+420·sin(1.6t)、y = 270+250·cos(1.1t)，t 是這個場景自己的時間
tl0, tl1 = N0 / FPS - 4, (N0 + 1) / FPS - 4
exp_dx = (420 * np.sin(1.6 * tl1) - 420 * np.sin(1.6 * tl0)) / 2
exp_dy = (250 * np.cos(1.1 * tl1) - 250 * np.cos(1.1 * tl0)) / 2
med = np.median(mv.reshape(-1, 2), axis=0)
print(f"MV 中位數 (dx, dy) = ({med[1]:.0f}, {med[0]:.0f})；由 crop 運算式推得的理論值 ≈ ({exp_dx:.1f}, {exp_dy:.1f})（960×540 尺度）")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
axes[0].imshow(cur, cmap="gray")
yy, xx = np.mgrid[0:mv.shape[0], 0:mv.shape[1]] * 16 + 8
axes[0].quiver(xx, yy, mv[..., 1], mv[..., 0], color="yellow", angles="xy", scale_units="xy", scale=0.5, width=0.003)
axes[0].set_title("Motion vectors (16x16 blocks, search +/-8)")
axes[1].imshow(np.abs(cur - prev_crop), cmap="magma", vmin=0, vmax=120)
axes[1].set_title(f"|cur - prev|  energy={e_no_mc:.2e}")
axes[2].imshow(np.abs(cur - pred), cmap="magma", vmin=0, vmax=120)
axes[2].set_title(f"|cur - motion-compensated|  energy={e_mc:.2e}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

**觀察**

- 大部分箭頭方向、長度一致，中位數 MV 和由 `crop` 運算式算出的理論值吻合（差一點點來自 yuv420p 的 crop 對齊到偶數像素與縮放）：整個畫面在平移。
- 做了動作補償後，殘差能量只剩一小部分——要編碼的東西少了很多，這就是時間冗餘被拿掉的地方。殘差圖上剩下的亮點，多半是**自己在動的物體**（testsrc2 的移動方塊、彩色線條）或 MV 搜尋範圍不夠的區塊。
- 真實編碼器還會做：1/4 像素精度、可變區塊大小（16×16 到 4×4，H.265 更大）、從**多張**參考影格找、快速搜尋（不是 289 個位置都試），以及用 rate-distortion 選擇最划算的 MV。

## 13.12–13.13 I／P／B 影格與 GOP

**概念**

| 影格 | 參考誰 | 大小 | 特性 |
|---|---|---|---|
| **I**（Intra）| 不參考別張，只用畫面內預測 | 最大 | 可以**單獨解碼**：拖曳時間軸、剪輯的起點都需要它 |
| **P**（Predicted）| 參考**之前**已解碼的影格 | 中 | 存 MV＋殘差 |
| **B**（Bi-directional）| 可參考**之前與之後**的影格 | 最小 | 每個區塊**各自選擇**用前、後或兩者加權來預測，再加上殘差——**不是**把前後兩張「平均」 |

- 因為 B 影格要參考「之後」的影格，**解碼順序 ≠ 顯示順序**：檔案裡是 `I P B B`，畫面上是 `I B B P`。這也是為什麼影片需要 PTS（顯示時間）與 DTS（解碼時間）兩種時間戳。
- **GOP**（Group of Pictures）：從一個 I 影格（關鍵影格）到下一個 I 影格之前的一組影格。`-g 30` 表示最多每 30 張一個關鍵影格。
  - GOP 短 → **隨機存取（拖曳、跳轉）快**、出錯時影響範圍小，但 I 影格多 → 檔案大。
  - GOP 長 → 檔案小，但跳到某個時間點時，要從前一個關鍵影格開始**一路解碼**到目標。
- x264 預設會在**場景切換**時自動插入 I 影格（scenecut），即使 GOP 還沒到上限。

**工具：PyAV**（BSD-3-Clause）是 FFmpeg 函式庫的 Python 綁定。`container.demux()` 一個個拿出**壓縮後的 packet**（`packet.size` 就是那一格佔多少 bytes、`packet.is_keyframe`），`packet.decode()` 解出**影格**（`frame.pict_type` 是 I／P／B）。

> PyAV 的 API 會隨版本改變：`frame.pict_type` 在舊版是一個 enum（`.name` 為 `'I'`），在新版（例如 av 14 以後）可能是整數（1＝I、2＝P、3＝B）。下面的 `pict_letter()` 兩種都處理。

> **先預測再執行**：`-g 30 -bf 2`、`-g 250`、`-bf 0` 三種設定，哪一個檔案最大？哪一個 I 影格最多？`-g 250` 在 10 秒（300 張）的影片裡只會有 1 個 I 影格嗎？

In [ ]:
# 共同參考：把 input.mp4 縮成 640×360 的 FFV1 無損檔，用來算各版本的 PSNR（13.20 也會用到）
run(["ffmpeg", "-i", "input.mp4", "-an", "-vf", "scale=640:360", "-c:v", "ffv1", "ref_360_10s.nut"])
GOP_VARIANTS = {
    "g30_bf2": ["-g", "30", "-bf", "2"],
    "g250": ["-g", "250"],
    "bf0": ["-bf", "0"],
}
gop_files = {}
for name, opts in GOP_VARIANTS.items():
    out = f"gop_{name}.mp4"
    cmd = ["ffmpeg", "-i", "input.mp4", "-vf", "scale=640:360", "-an", "-c:v", "libx264", "-crf", "23",
           "-preset", "medium", *opts, out]
    _, sec = run(cmd)
    gop_files[name] = out
    print(f"{shlex.join(cmd[:1] + cmd[1:])}   ({sec:.1f} s)")

In [ ]:
PICT = {0: "?", 1: "I", 2: "P", 3: "B", 4: "S", 5: "SI", 6: "SP", 7: "BI"}


def pict_letter(pt):
    # 相容不同 PyAV 版本：字串、enum（有 .name）、整數
    if isinstance(pt, str):
        return pt
    name = getattr(pt, "name", None)
    if isinstance(name, str) and name not in ("", "NONE"):
        return name
    try:
        return PICT.get(int(pt), "?")
    except (TypeError, ValueError):
        return "?"


def analyze_gop_pyav(path):
    # 回傳 (frames：顯示順序的 DataFrame, decode_order：packet 的 pts 依檔案順序)
    import av
    rows, sizes, decode_order = [], {}, []
    with av.open(str(path)) as c:
        s = c.streams.video[0]
        tb = s.time_base
        for pkt in c.demux(s):
            if pkt.pts is not None and pkt.size:
                sizes[pkt.pts] = (pkt.size, bool(pkt.is_keyframe))
                decode_order.append(pkt.pts)
            for fr in pkt.decode():
                rows.append(dict(pts=fr.pts, type=pict_letter(fr.pict_type)))
    df = pd.DataFrame(rows).sort_values("pts").reset_index(drop=True)
    df["time"] = df.pts.astype(float) * float(tb)
    df["size"] = df.pts.map(lambda p: sizes.get(p, (np.nan, False))[0])
    df["keyframe"] = df.pts.map(lambda p: sizes.get(p, (np.nan, False))[1])
    return df, decode_order


def analyze_gop_ffprobe(path):
    # 替代方案：沒有 PyAV 時用 ffprobe 的 frame 資訊（欄位名稱依 ffprobe 版本可能不同）
    fr = probe(path, "frame=pict_type,key_frame,pkt_size,best_effort_timestamp_time", select="v:0")["frames"]
    df = pd.DataFrame(dict(type=[f.get("pict_type", "?") for f in fr],
                           time=[to_float(f.get("best_effort_timestamp_time")) for f in fr],
                           size=[int(f.get("pkt_size", 0) or 0) for f in fr],
                           keyframe=[str(f.get("key_frame")) == "1" for f in fr]))
    return df, None


def gop_string(types, width=60):
    s = "".join(types)
    return "\n".join(s[i:i + width] for i in range(0, len(s), width))


GOP = {}
for name, path in gop_files.items():
    try:
        GOP[name] = analyze_gop_pyav(path)
        STATUS["PyAV GOP 分析"] = "real"
    except Exception as err:
        print(f"PyAV 失敗（{type(err).__name__}: {err}）→ 改用 ffprobe")
        GOP[name] = analyze_gop_ffprobe(path)
        STATUS["PyAV GOP 分析"] = "fallback"

summary = []
for name, (df, _) in GOP.items():
    print(f"\n=== {name}（前 90 張，顯示順序，每列 30 張）===")
    print(gop_string(df.type.head(90), width=30))
    key_t = df.loc[df.type == "I", "time"].round(2).tolist()
    summary.append(dict(variant=name, frames=len(df), I=int((df.type == "I").sum()), P=int((df.type == "P").sum()),
                        B=int((df.type == "B").sum()), size_KB=round(Path(gop_files[name]).stat().st_size / 1e3, 1),
                        PSNR_dB=quality(gop_files[name], "ref_360_10s.nut", metrics=("psnr",))["PSNR_dB"],
                        mean_I_KB=round(df.loc[df.type == "I", "size"].mean() / 1e3, 2),
                        mean_P_KB=round(df.loc[df.type == "P", "size"].mean() / 1e3, 2),
                        mean_B_KB=round(df.loc[df.type == "B", "size"].mean() / 1e3, 2) if (df.type == "B").any() else None,
                        I_frame_times_s=key_t if len(key_t) <= 12 else key_t[:12] + ["…"]))
gop_df = pd.DataFrame(summary)
display(gop_df)

In [ ]:
fig, axes = plt.subplots(len(GOP), 1, figsize=(12, 2.4 * len(GOP)), sharex=True)
for ax, (name, (df, _)) in zip(np.atleast_1d(axes), GOP.items()):
    ax.bar(df.index, df["size"] / 1e3, width=1.0, color=[TYPE_COLORS.get(t, "#7f7f7f") for t in df.type])
    for _, t0, _ in SCENES[1:]:
        ax.axvline(t0 * FPS, color="k", ls="--", lw=0.8)
    ax.set_yscale("log")
    ax.set_ylabel("KB / frame")
    ax.set_title(f"{name}: frame size by type (red=I, blue=P, green=B; dashed=scene change)", fontsize=10)
axes[-1].set_xlabel("Frame index (display order)")
plt.tight_layout()
plt.show()

df, dec = GOP["g30_bf2"]
if dec is not None:
    t_of = dict(zip(df.pts, df.type))
    print("解碼順序（檔案中 packet 的順序）：", "".join(t_of.get(p, "?") for p in dec[:16]))
    print("顯示順序（依 PTS 排序）：        ", "".join(df.type.head(16)))

# 隨機存取：想從 5.5 s 開始播，要從前一個 I 影格開始解碼幾張？
target = int(5.5 * FPS)
for name, (df, _) in GOP.items():
    prev_i = df.index[(df.type == "I") & (df.index <= target)].max()
    print(f"{name:8s}：跳到 5.5 s（第 {target} 張）要從第 {prev_i} 張（{prev_i / FPS:.2f} s 的 I 影格）解碼 {target - prev_i + 1} 張")

**等價的 ffprobe 指令**（講義寫法，不需要 Python）：

```bash
ffprobe -v error -select_streams v:0 -show_entries frame=pict_type,key_frame,pkt_size -of csv input.mp4
```

注意兩件事：(1) csv 欄位的順序是 **ffprobe 內部的順序**（`key_frame`、`pkt_size`、`pict_type`），**不是**你在 `-show_entries` 裡寫的順序；(2) 有些影格後面會多一個空欄位，那是 side data。要用程式處理時，改成 `-of json` 比較保險。下面先原樣執行講義指令，再用 JSON 版本和 PyAV 的結果交叉比對。

In [ ]:
FFPROBE_FRAMES = "ffprobe -v error -select_streams v:0 -show_entries frame=pict_type,key_frame,pkt_size -of csv input.mp4"
p, _ = run(FFPROBE_FRAMES, show=True)
lines = p.stdout.splitlines()
print("\n".join(lines[:6]), f"\n…（共 {len(lines)} 行）")

fr = probe(gop_files["g30_bf2"], "frame=pict_type,key_frame,pkt_size", select="v:0")["frames"]
ff_types = "".join(f.get("pict_type", "?") for f in fr)
py_types = "".join(GOP["g30_bf2"][0].type)
print(f"\ngop_g30_bf2.mp4：ffprobe 與 PyAV 的 I/P/B 序列{'完全相同' if ff_types == py_types else '不同（請檢查版本）'}"
      f"（{len(ff_types)} 張）；ffprobe 的 pkt_size 總和 = {sum(int(f.get('pkt_size', 0) or 0) for f in fr):,} bytes")

**觀察**

- I 影格比 P 大很多、P 又比 B 大（注意 y 軸是對數）。
- `g30_bf2` 每 30 張就有一個 I，PSNR 和 `g250` 幾乎一樣，檔案卻大了將近一倍——多出來的幾乎都是 I 影格。`g250` 雖然上限是 250，但在 4 s、7 s 的**場景切換處自動插入了 I 影格**（scenecut）。
- `bf0` 完全沒有 B。在這支**合成**素材上，它的檔案和 `g250` 差不多、PSNR 甚至略高；在一般攝影素材上，B 影格通常能省下一些位元，但效果**依內容而定**（這裡的靜態投影片幾乎沒有殘差、高動態場景是固定紋理的平移，P 影格就預測得很好）。所以要用實驗驗證，而且要**同時看大小與品質**：用 CRF 時，各版本的品質不一定相同。
- 解碼順序是 `I P B B …`、顯示順序是 `I B B P …`：B 影格要等「後面」的 P 先解出來。
- GOP 越長，跳轉到某個時間點需要先解碼的影格越多——這就是線上影片、直播常用 1–2 秒 GOP 的原因。

## 13.14 靜態場景 vs 複雜場景（＋ PySceneDetect 場景偵測）

**概念**：用 CRF（固定品質）編碼時，編碼器會**依內容分配位元**：靜態投影片的影格幾乎都能從前一張預測出來，殘差接近 0；快速平移＋細碎紋理的畫面，每格都有大量殘差。所以**同一支影片內，位元率會隨場景大幅變動**。

下面用 ffprobe 讀出 `input.mp4` 每個 packet 的大小，加總成「每秒位元率」。

> **先預測再執行**：三個場景的每秒位元率，誰最高、誰最低？差幾倍？

In [ ]:
def bitrate_per_second(path, bin_s=1.0):
    pk = probe(path, "packet=pts_time,size", select="v:0")["packets"]
    t = np.array([to_float(p.get("pts_time")) for p in pk], dtype=float)
    size = np.array([int(p.get("size", 0)) for p in pk], dtype=float)
    ok = ~np.isnan(t)
    bins = np.floor(t[ok] / bin_s).astype(int)
    mbps = np.bincount(bins, weights=size[ok]) * 8 / 1e6 / bin_s
    return np.arange(len(mbps)) * bin_s, mbps


sec_t, sec_mbps = bitrate_per_second("input.mp4")
fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(sec_t + 0.5, sec_mbps, width=0.9, color="#4c72b0")
for name, t0, t1 in SCENES:
    ax.text((t0 + t1) / 2, max(sec_mbps) * 1.02, name, ha="center", fontsize=9)
    ax.axvline(t0, color="k", ls="--", lw=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Video bitrate (Mbps)")
ax.set_ylim(0, max(sec_mbps) * 1.15)
ax.set_title("input.mp4 (libx264 CRF 18): bitrate per second")
plt.show()
scene_rate = {name: sec_mbps[t0:t1].mean() for name, t0, t1 in SCENES}
for k, v in scene_rate.items():
    print(f"{k:20s} 平均 {v:6.2f} Mbps")
print("（第 0、2、4… 秒內含關鍵影格，所以這幾秒會比同場景其他秒略高）")

### 工具：PySceneDetect 自動找出鏡頭切換

[PySceneDetect](https://www.scenedetect.com/)（BSD-3-Clause）逐格比較畫面，找出**場景切換點（cut）**：

- `ContentDetector`：比較相鄰影格的 HSV 色彩與亮度差異，超過門檻（預設 27）就算切換——適合「硬切」。
- `AdaptiveDetector`：和「附近幾格的平均變化」比較，在**畫面一直在動**（例如手持運鏡）時比較不會誤判，但可能漏掉相似場景間的切換。

偵測結果就是一份**剪輯點清單**：下週做時間軸剪輯時，可以直接拿來切段、加轉場或對字幕。這也呼應 13.12：x264 在這些點會自動插入 I 影格。

> **先預測再執行**：我們知道真正的切換點在 4 s 與 7 s。兩種偵測器都找得到嗎？

In [ ]:
def tc_seconds(tc):
    v = getattr(tc, "seconds", None)                 # scenedetect 0.7+：屬性
    return float(v) if v is not None and not callable(v) else float(tc.get_seconds())


def tc_frame(tc):
    v = getattr(tc, "frame_num", None)
    return int(v) if v is not None and not callable(v) else int(tc.get_frames())


SCENE_LISTS = {}
warnings.filterwarnings("ignore", message=".*IProgress.*")     # 沒有 ipywidgets 時 tqdm 的無害警告
try:
    from scenedetect import AdaptiveDetector, ContentDetector, detect
    for name, det in [("ContentDetector", ContentDetector()), ("AdaptiveDetector", AdaptiveDetector())]:
        t0 = time.perf_counter()
        scenes = detect("input.mp4", det)
        SCENE_LISTS[name] = [(tc_seconds(a), tc_seconds(b), tc_frame(a), tc_frame(b)) for a, b in scenes]
        print(f"{name}：{len(scenes)} 個場景（{time.perf_counter() - t0:.1f} s）")
    STATUS["PySceneDetect"] = "real"
except Exception as err:
    print(f"PySceneDetect 無法使用（{type(err).__name__}: {err}）→ 改用 FFmpeg 的 select='gt(scene,0.3)'")
    p, _ = run(["ffmpeg", "-hide_banner", "-i", "input.mp4", "-vf", "select='gt(scene,0.3)',showinfo",
                "-f", "null", "-"], quiet=False)
    cuts = [0.0] + [float(x) for x in re.findall(r"pts_time:([\d.]+)", p.stderr)] + [float(DUR)]
    SCENE_LISTS["ffmpeg scene score"] = [(a, b, round(a * FPS), round(b * FPS)) for a, b in zip(cuts[:-1], cuts[1:])]
    STATUS["PySceneDetect"] = "fallback"

for name, lst in SCENE_LISTS.items():
    print(f"\n{name}")
    display(pd.DataFrame(lst, columns=["start_s", "end_s", "start_frame", "end_frame"]).assign(
        duration_s=lambda d: d.end_s - d.start_s))
print("真正的切換點：", [t0 for _, t0, _ in SCENES[1:]], "s")

**觀察**

- 靜態投影片的位元率只有高動態場景的一小部分：**同樣 CRF，不同內容，位元率可以差一個數量級**。所以「這支影片 10 Mbps」本身說明不了品質。
- `ContentDetector` 找到 4 s 與 7 s 兩個切換點；`AdaptiveDetector` 可能把後兩個場景合併——因為兩個場景本來就都「一直在大幅變動」，相對變化不明顯。偵測器與門檻要依素材調整（`ContentDetector(threshold=...)`），**自動結果要人工確認**。

## 13.15 Stream Copy vs 重新編碼

**概念**：剪一段影片有兩種做法：

| | Stream Copy（`-c copy`）| 重新編碼（re-encode）|
|---|---|---|
| 做法 | 直接搬 packet，不解碼 | 解碼 → 剪 → 再編碼 |
| 速度 | 幾乎瞬間 | 慢（和編碼一樣）|
| 畫質 | 完全不變 | 再壓一次，會有世代損失 |
| 切點精準度 | **只能從關鍵影格開始**（13.12：P／B 影格無法單獨解碼）| 可以精準到任何一格 |

講義的指令是 `ffmpeg -ss 10 -i input.mp4 -t 5 -c copy clip_copy.mp4`（從第 10 秒剪 5 秒）。我們的 `input.mp4` 只有 10 秒，所以改成**從 5 秒剪 4 秒**；`input.mp4` 的關鍵影格在 0、2、4、6、8 s，**5 s 不是關鍵影格**。

> **先預測再執行**：`clip_copy.mp4` 的長度、檔案大小、第一張畫面會和 `clip_reenc.mp4` 一樣嗎？

In [ ]:
def keyframe_times(path):
    p, _ = run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-skip_frame", "nokey",
                "-show_entries", "frame=best_effort_timestamp_time", "-of", "csv=p=0", str(path)])
    return [round(float(x.strip(",")), 3) for x in p.stdout.split() if x.strip(",")]


def cut_compare(src, start, dur, tag="clip"):
    src_keys = keyframe_times(src)
    copy_cmd = f"ffmpeg -ss {start} -i {src} -t {dur} -c copy {tag}_copy.mp4"
    reenc_cmd = (f"ffmpeg -ss {start} -i {src} -t {dur} -c:v libx264 -crf 18 -preset veryfast "
                 f"-c:a aac -b:a 128k {tag}_reenc.mp4")
    rows = []
    for kind, cmd, out in [("stream copy", copy_cmd, f"{tag}_copy.mp4"), ("re-encode", reenc_cmd, f"{tag}_reenc.mp4")]:
        _, sec = run(cmd, show=True)
        s = probe(out, "stream=duration,nb_frames:format=duration,size,start_time", select="v:0")
        pk = probe(out, "packet=pts_time,flags", select="v:0")["packets"]
        p2, _ = run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-count_frames",
                     "-show_entries", "stream=nb_read_frames", "-of", "csv=p=0", out])
        first = probe(out, "frame=pict_type,best_effort_timestamp_time", select="v:0")["frames"][0]
        rows.append(dict(method=kind, time_s=round(sec, 2), size_MB=round(int(s["format"]["size"]) / 1e6, 2),
                         duration_s=to_float(s["format"].get("duration")),
                         video_packets=len(pk), frames_shown=int(p2.stdout.strip().strip(",") or 0),
                         first_packet_pts_s=to_float(pk[0].get("pts_time")), first_packet_flags=pk[0].get("flags"),
                         first_shown_frame_type=first.get("pict_type")))
    print(f"\n來源關鍵影格時間（s）：{src_keys}")
    return pd.DataFrame(rows)


CUT_START, CUT_DUR = 5, 4
cut_df = cut_compare("input.mp4", CUT_START, CUT_DUR)
display(cut_df)

In [ ]:
# 第一張畫面：一般播放（遵守 MP4 的 edit list） vs 忽略 edit list（看見檔案裡真正的第一張）
views = {
    "source @ 5.0 s": grab("input.mp4", 5.0, size=(480, 270)),
    "clip_copy first frame": grab("clip_copy.mp4", 0, size=(480, 270)),
    "clip_copy, -ignore_editlist 1": grab("clip_copy.mp4", 0, size=(480, 270), extra_input_opts=("-ignore_editlist", "1")),
    "source @ 4.0 s (keyframe)": grab("input.mp4", 4.0, size=(480, 270)),
}
fig, axes = plt.subplots(1, 4, figsize=(15, 2.6))
for ax, (lab, img) in zip(axes, views.items()):
    ax.imshow(img)
    ax.set_title(lab, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
hidden = views["clip_copy, -ignore_editlist 1"].astype(float)
print(f"忽略 edit list 的第一張 vs 來源 4.0 s：平均差 {np.abs(hidden - views['source @ 4.0 s (keyframe)']).mean():.2f}；"
      f"vs 來源 5.0 s：{np.abs(hidden - views['source @ 5.0 s']).mean():.2f}")

**觀察**

- Stream copy 不到 0.1 秒就完成，重新編碼要好幾秒。
- `clip_copy.mp4` 的 packet 比實際顯示的影格**多了約 30 個**，而且第一個 packet 的時間是**負的**（−1.0 s，旗標 `K` = keyframe）：FFmpeg 只能從 **4.0 s 的關鍵影格**開始複製，再用 MP4 的 **edit list** 把多出來的 1 秒「藏起來」。檔案因此比較大，長度也不是剛好 4.000 s。
- 遵守 edit list 的播放器會從 5.0 s 的畫面開始（第一張顯示的影格甚至是 B 影格，因為它的參考影格就藏在前面）；**忽略** edit list 的播放器或剪輯軟體，會從 4.0 s 開始播——切點「往前跑」了 1 秒。不同 FFmpeg 版本、不同播放器的行為也可能不一樣，所以用 `-c copy` 剪輯後**一定要檢查**。
- 重新編碼版本精準從 5.0 s 開始、第一張是 I 影格、長度剛好 4 s。
- GUI 工具 **LosslessCut**（GPL-2.0）就是 `-c copy` 剪輯的圖形介面：無損、極快，但它的說明也提醒：在關鍵影格剪輯模式下，**起點會被移到前一個關鍵影格**。

## 13.16–13.17 Codec、Encoder、Container；Mux 與 Demux

| 名詞 | 是什麼 | 例子 |
|---|---|---|
| **Codec**（編碼格式／標準）| 位元串流**長什麼樣**的規格 | H.264／AVC、H.265／HEVC、AV1、VP9、AAC、Opus |
| **Encoder**（編碼器）| 把影像變成該格式位元串流的**實作** | H.264 的 `libx264`、`h264_nvenc`（NVIDIA GPU）、OpenH264；AV1 的 `libsvtav1`、`libaom-av1` |
| **Decoder**（解碼器）| 把位元串流還原成影像的實作 | FFmpeg 內建 `h264`、`dav1d`、手機的硬體解碼器 |
| **Container**（容器）| 把影像、聲音、字幕串流**裝在一起**，加上時間戳、索引、中繼資料 | MP4、MOV、MKV、WebM、MPEG-TS |

- 同一個 codec 可以有很多 encoder，**品質與速度可能差很多**（例如同樣是 H.264，libx264 與硬體編碼器）。
- **Mux**（multiplex）：把多條串流裝進容器；**Demux**：把串流拆出來。只換容器（remux）配 `-c copy`，**位元串流完全不變**。
- 副檔名只是名字：`.mp4` **不保證**裡面是 H.264；把 `.mp4` 改名成 `.mkv` **不會轉檔**。
- 容器有支援範圍：例如 WebM 只接受 VP8／VP9／AV1 與 Vorbis／Opus，把 H.264＋AAC `-c copy` 進 `.webm` 會失敗。

> **先預測再執行**：把影像和聲音拆開再裝回 MKV、MP4，影像位元串流的 MD5 會變嗎？

In [ ]:
def stream_md5(path, sel="0:v:0"):
    # 只對「壓縮後的 packet」算 MD5（-c copy，不解碼）：用來確認位元串流是否完全相同
    p, _ = run(["ffmpeg", "-i", str(path), "-map", sel, "-c", "copy", "-f", "md5", "-"])
    return p.stdout.strip().replace("MD5=", "")[:12]


cmds = [
    "ffmpeg -i input.mp4 -map 0:v:0 -c copy video_only.mp4",             # demux：只取影像
    "ffmpeg -i input.mp4 -map 0:a:0 -c copy audio_only.m4a",             # demux：只取聲音
    "ffmpeg -i video_only.mp4 -i audio_only.m4a -map 0:v -map 1:a -c copy remux.mkv",   # mux 成 MKV
    "ffmpeg -i video_only.mp4 -i audio_only.m4a -map 0:v -map 1:a -c copy remux.mp4",   # mux 回 MP4
]
for c in cmds:
    _, sec = run(c, show=True)

shutil.copy("input.mp4", "renamed.mkv")                                  # 只改副檔名
rows = []
for f in ["input.mp4", "video_only.mp4", "audio_only.m4a", "remux.mkv", "remux.mp4", "renamed.mkv"]:
    d = info(f)
    has_v = d["vcodec"] is not None
    rows.append(dict(file=f, container=d["container"], vcodec=d["vcodec"], acodec=d["acodec"],
                     size_MB=round(d["size_MB"], 3), video_md5=stream_md5(f) if has_v else "—"))
mux_df = pd.DataFrame(rows)
display(mux_df)

p, _ = run("ffmpeg -i input.mp4 -c copy bad.webm", check=False, show=True)
print("→ 失敗（預期中）：", (p.stderr.strip().splitlines() or ["?"])[0][:160])

In [ ]:
# .mp4 不保證是 H.264：做幾支「副檔名是 .mp4、但 codec 不同」的檔案
other = [("mpeg4", ["-c:v", "mpeg4", "-q:v", "5"], "not_h264_mpeg4.mp4"),
         ("libx265", ["-c:v", "libx265", "-crf", "28", "-tag:v", "hvc1", "-x265-params", "log-level=error"], "not_h264_hevc.mp4"),
         ("libsvtav1", ["-c:v", "libsvtav1", "-crf", "35", "-preset", "10"], "not_h264_av1.mp4"),
         ("libaom-av1", ["-c:v", "libaom-av1", "-crf", "35", "-b:v", "0", "-cpu-used", "8", "-row-mt", "1"], "not_h264_av1.mp4")]
rows, made_av1 = [], False
for enc, opts, out in other:
    if enc not in ENCODERS or (out == "not_h264_av1.mp4" and made_av1):
        continue
    p, sec = run(["ffmpeg", "-t", "2", "-i", "input.mp4", "-vf", "scale=640:360", "-an", *opts, out], check=False)
    if p.returncode != 0:
        print(f"{enc} 失敗，略過：{p.stderr.strip()[-150:]}")
        continue
    made_av1 |= out == "not_h264_av1.mp4"
    d = probe(out, "stream=codec_name,codec_long_name,codec_tag_string:format=format_name", select="v:0")
    rows.append(dict(file=out, encoder_used=enc, format_name=d["format"]["format_name"],
                     codec_name=d["streams"][0]["codec_name"], codec_tag=d["streams"][0].get("codec_tag_string"),
                     codec_long_name=d["streams"][0].get("codec_long_name")))
display(pd.DataFrame(rows))

**觀察**

- `video_only.mp4`、`remux.mkv`、`remux.mp4`、`renamed.mkv` 的**影像 MD5 全部和 `input.mp4` 一樣**：demux／remux 只是換盒子，裡面的 H.264 位元串流一個 bit 都沒變（所以也幾乎不花時間）。容器大小略有差異，來自各容器自己的索引與中繼資料。
- `renamed.mkv` 的 `format_name` 仍然是 `mov,mp4,…`：**改副檔名不會轉檔**，ffprobe 看的是檔案內容，不是檔名（反過來，有些播放器只看副檔名，就會打不開）。
- 同樣叫 `.mp4`，裡面可以是 MPEG-4 Part 2、HEVC、AV1……所以要知道一支影片「是什麼」，**一定要用 ffprobe／MediaInfo 看**（13.22）。

## 13.18 Codec 比較：H.264／H.265／AV1（＋VP9）

| Codec | 年代 | 壓縮效率（同畫質）| 編碼速度 | 解碼相容性 | 授權 |
|---|---|---|---|---|---|
| H.264／AVC | 2003 | 基準 | 快 | **幾乎所有裝置、瀏覽器** | 專利授權 |
| H.265／HEVC | 2013 | 約省 25–50% | 較慢 | 手機、電視硬體多半支援；瀏覽器支援不一 | 專利授權較複雜 |
| VP9 | 2013 | 與 HEVC 相近 | 慢 | YouTube、Chrome、Android | 免授權費（Google）|
| AV1 | 2018 | 比 HEVC 再省一些 | 最慢（軟體）| 新手機、新 GPU、主流瀏覽器 | 免授權費（AOMedia）|

下面先用 `ffmpeg -hide_banner -encoders` 檢查這台機器**有哪些編碼器**，沒有的就跳過。為了讓 CPU 跑得完，用 640×360、4 秒（2–6 s，跨越靜態與高動態場景）的 **FFV1 無損**檔當共同來源與參考。

> 小插曲（也是容器的一課）：MKV、WebM 的時間戳精度是 1 毫秒，30 fps 的影格時間會被四捨五入成 0、33、67 ms…；MP4、NUT 則能保留 1/30 秒的時間基準。FFmpeg 的 `psnr`／`ssim` 濾鏡是**依時間戳配對**兩邊的影格，兩邊時間基準不同時可能「錯一格比較」，PSNR 就會嚴重偏低（本來 42 dB 會變成 29 dB）。所以參考檔用 **NUT** 容器，而 `quality()` 也先把兩邊依影格順序重新編號再比較。

**注意：不同 codec 的 CRF 數值不能直接比較**——x264 的 CRF 23、x265 的 CRF 28、AV1 的 CRF 35 只是各自的「常用預設附近」，不代表同樣品質。公平比較要固定品質看大小，或固定大小看品質（業界用 BD-rate 曲線）；也要說明 preset（速度）設定。

> **先預測再執行**：誰的檔案最小？誰編碼最慢？

In [ ]:
p, _ = run("ffmpeg -hide_banner -encoders", quiet=False)
wanted = ["libx264", "libx265", "libsvtav1", "libaom-av1", "libvpx-vp9"]
print("\n".join(ln for ln in p.stdout.splitlines() if any(f" {w} " in ln for w in wanted)))

run(["ffmpeg", "-ss", "2", "-t", "4", "-i", "input.mp4", "-an", "-vf", "scale=640:360", "-c:v", "ffv1", "ref_360.nut"])
ref_mb = Path("ref_360.nut").stat().st_size / 1e6
print(f"\n參考檔 ref_360.nut（FFV1 無損）：{ref_mb:.2f} MB")

CODECS = [
    ("H.264", "libx264", ["-crf", "23", "-preset", "medium"], "mp4", None),
    ("H.265", "libx265", ["-crf", "28", "-preset", "medium", "-tag:v", "hvc1", "-x265-params", "log-level=error"], "mp4", None),
    ("AV1 (SVT-AV1)", "libsvtav1", ["-crf", "35", "-preset", "8"], "mp4", ["-qp", "35", "-preset", "8"]),
    ("AV1 (libaom)", "libaom-av1", ["-crf", "35", "-b:v", "0", "-cpu-used", "8", "-row-mt", "1"], "mp4", None),
    ("VP9", "libvpx-vp9", ["-crf", "33", "-b:v", "0", "-deadline", "good", "-cpu-used", "5", "-row-mt", "1"], "webm", None),
]
codec_rows = []
for label, enc, opts, ext, alt in CODECS:
    if enc not in ENCODERS:
        STATUS[f"編碼器 {enc}"] = "skipped"
        print(f"{label}：這個 FFmpeg 沒有 {enc}，略過")
        continue
    out = f"codec_{enc}.{ext}"
    p, sec = run(["ffmpeg", "-i", "ref_360.nut", "-c:v", enc, *opts, out], check=False)
    if p.returncode != 0 and alt:                      # 舊版 FFmpeg 的參數名稱不同時再試一次
        p, sec = run(["ffmpeg", "-i", "ref_360.nut", "-c:v", enc, *alt, out], check=False)
    if p.returncode != 0:
        STATUS[f"編碼器 {enc}"] = "skipped"
        print(f"{label}：編碼失敗，略過（{p.stderr.strip()[-150:]}）")
        continue
    STATUS[f"編碼器 {enc}"] = "real"
    _, dec = run(["ffmpeg", "-i", out, "-f", "null", "-"])
    q = quality(out, "ref_360.nut")
    d = info(out)
    codec_rows.append(dict(codec=label, encoder=enc, settings=" ".join(opts[:4]), size_KB=round(d["size_MB"] * 1e3, 1),
                           kbps=round(d["bitrate_Mbps"] * 1e3), encode_s=round(sec, 2), decode_s=round(dec, 2),
                           PSNR_dB=q["PSNR_dB"], SSIM=q["SSIM"]))
CODEC_RESULTS = pd.DataFrame(codec_rows)
display(CODEC_RESULTS)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for _, r in CODEC_RESULTS.iterrows():
    axes[0].scatter(r.size_KB, r.PSNR_dB, s=60)
    axes[0].annotate(r.codec, (r.size_KB, r.PSNR_dB), textcoords="offset points", xytext=(5, 4), fontsize=9)
axes[0].set_xlabel("File size (KB)")
axes[0].set_ylabel("PSNR (dB)")
axes[0].set_title("Size vs quality (upper-left is better)")
axes[1].barh(CODEC_RESULTS.codec, CODEC_RESULTS.encode_s, color="#dd8452")
axes[1].set_xlabel("Encode time (s)")
axes[1].set_title("Encode time (640x360, 4 s, CPU)")
plt.tight_layout()
plt.show()

**觀察**

- 新一代 codec（H.265、AV1）通常能用更小的檔案得到相近或更高的 PSNR，但代價是**編碼時間**與**相容性**。
- 這裡每個 codec 只用了**一個** CRF，所以只能說「在這組設定下」誰大誰小；要說「AV1 比 H.264 省 X%」，需要每個 codec 掃過多個 CRF、畫出大小–品質曲線再比較。
- 交作業、給一般觀眾看：H.264＋AAC＋MP4＋yuv420p 仍是最不會出錯的組合；自己存檔或上傳平台（平台會再轉檔）時才考慮 H.265／AV1。

## 13.20–13.21 位元率控制（CBR／VBR／CRF）與 Preset

| 模式 | 你指定什麼 | 結果 | 適合 |
|---|---|---|---|
| **CBR**（固定位元率）| 位元率，且不准超過 | 每秒資料量幾乎固定；簡單畫面浪費、複雜畫面不夠 | 直播、廣播（頻寬固定）|
| **VBR／ABR**（變動／平均位元率）| 目標平均位元率（可 2-pass）| 複雜處多給、簡單處少給，**檔案大小可預測** | 有檔案大小限制時 |
| **CRF**（Constant Rate Factor，固定品質）| 品質等級（x264：0–51，預設 23，**越小越好、越大檔**）| 品質大致固定，**檔案大小不可預測** | 存檔、上傳、作業（最常用）|

講義的基本指令（CRF 23、preset medium、聲音轉 AAC 128 kbps）：

```bash
ffmpeg -i input.mp4 -c:v libx264 -crf 23 -preset medium -c:a aac -b:a 128k output.mp4
```

**Preset** 控制編碼器「多努力找最好的壓縮方式」（動作搜尋範圍、參考影格數、區塊模式……）：`ultrafast` … `medium`（預設）… `slow` … `veryslow`。**同樣 CRF 下，preset 越慢 → 檔案越小（或同大小品質更好），但編碼時間更長**；preset 不影響解碼相容性。HandBrake 的「RF」就是 x264／x265 的 CRF，它的 Presets 則是整組設定（解析度、codec、RF、音訊）。

> **先預測再執行**：(1) `output.mp4`（聲音轉 AAC）和 13.25 的 `crf23.mp4`（聲音 `-c:a copy`）大小差多少？(2) CRF、2-pass VBR、CBR 用**同樣的平均位元率**時，每秒位元率曲線會長什麼樣？誰的 PSNR 最高？(3) `ultrafast` 比 `slow` 快幾倍、大幾倍？

In [ ]:
BASE_CMD = "ffmpeg -i input.mp4 -c:v libx264 -crf 23 -preset medium -c:a aac -b:a 128k output.mp4"
_, base_sec = run(BASE_CMD, show=True)
display(pd.DataFrame([dict(info("output.mp4"), encode_s=round(base_sec, 1))]))

In [ ]:
# 位元率控制：640×360、10 秒。先用 CRF 23 編碼，量出它的平均位元率；
# 再讓 2-pass VBR 與 CBR 用「同樣的平均位元率」，比較位元怎麼分配、品質差多少
REF360 = "ref_360_10s.nut"                     # 13.12 建立的 FFV1 無損參考


def rc_encode(name, opts, out, two_pass=False):
    base = ["ffmpeg", "-i", REF360, "-c:v", "libx264", "-preset", "medium", *opts]
    sec = 0.0
    if two_pass:
        _, s1 = run(base + ["-pass", "1", "-passlogfile", "rc2pass", "-f", "null", os.devnull])
        _, s2 = run(base + ["-pass", "2", "-passlogfile", "rc2pass", out])
        sec = s1 + s2
    else:
        _, sec = run(base + [out])
    d = info(out)
    return dict(mode=name, size_KB=round(d["size_MB"] * 1e3), avg_kbps=round(d["bitrate_Mbps"] * 1e3),
                encode_s=round(sec, 2), PSNR_dB=quality(out, REF360, metrics=("psnr",))["PSNR_dB"])


rc_rows = [rc_encode("CRF 23", ["-crf", "23"], "rc_crf.mp4")]
K = rc_rows[0]["avg_kbps"]
print(f"CRF 23 的平均位元率 = {K} kbps → 以此作為 VBR 與 CBR 的目標")
rc_rows.append(rc_encode(f"VBR 2-pass {K}k", ["-b:v", f"{K}k"], "rc_vbr2pass.mp4", two_pass=True))
rc_rows.append(rc_encode(f"CBR {K}k", ["-b:v", f"{K}k", "-minrate", f"{K}k", "-maxrate", f"{K}k",
                                       "-bufsize", f"{K // 2}k", "-x264-params", "nal-hrd=cbr"], "rc_cbr.mp4"))
display(pd.DataFrame(rc_rows))

fig, ax = plt.subplots(figsize=(11, 3))
for r, f in zip(rc_rows, ["rc_crf.mp4", "rc_vbr2pass.mp4", "rc_cbr.mp4"]):
    t, mbps = bitrate_per_second(f)
    ax.step(np.append(t, t[-1] + 1), np.append(mbps, mbps[-1]) * 1000, where="post", label=r["mode"])
for _, t0, _ in SCENES[1:]:
    ax.axvline(t0, color="k", ls="--", lw=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Video bitrate (kbps)")
ax.set_title("Rate control at the same average bitrate: bitrate per second (dashed = scene change)")
ax.legend()
plt.show()

In [ ]:
# Preset：同一段高動態畫面（4–7 s，1080p）、同樣 CRF 23，只改 preset
PRESETS = ["ultrafast", "medium", "slow"]
preset_rows = []
for pr in PRESETS:
    out = f"preset_{pr}.mp4"
    _, sec = run(["ffmpeg", "-ss", "4", "-t", "3", "-i", "input.mp4", "-an", "-c:v", "libx264", "-crf", "23",
                  "-preset", pr, out])
    q = quality(out, "input.mp4", ref_opts=("-ss", "4", "-t", "3"))
    d = info(out)
    preset_rows.append(dict(preset=pr, encode_s=round(sec, 2), size_MB=round(d["size_MB"], 2),
                            bitrate_Mbps=round(d["bitrate_Mbps"], 2), **q))
PRESET_RESULTS = pd.DataFrame(preset_rows)
display(PRESET_RESULTS)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].bar(PRESET_RESULTS.preset, PRESET_RESULTS.encode_s, color="#dd8452")
axes[0].set_ylabel("Encode time (s)")
axes[1].bar(PRESET_RESULTS.preset, PRESET_RESULTS.size_MB, color="#4c72b0")
axes[1].set_ylabel("File size (MB)")
fig.suptitle("libx264 CRF 23: preset trade-off (3 s high-motion 1080p clip)")
plt.tight_layout()
plt.show()

**觀察**

- `output.mp4` 與 `crf23.mp4`（13.25）的影像設定相同，差別只在聲音（AAC 重新編碼 vs 複製）；聲音只佔總位元率很小一部分。
- 三個檔案大小幾乎一樣，但**位元分配方式**完全不同：CRF 與 2-pass VBR 跟著場景大起大落（靜態投影片幾乎不花位元，省下來給複雜場景）；CBR 每秒都差不多——靜態場景塞了用不到的位元（`nal-hrd=cbr` 會補填充資料），複雜場景卻不夠用，所以 PSNR 最低。
- 2-pass VBR 的大小可預測、品質接近 CRF；CRF 不用跑兩次，但事先不知道檔案會多大。
- Preset：`ultrafast` 最快但檔案大很多；`medium` 與 `slow` 檔案大小接近，`slow` 花了約兩倍時間換到略高的 PSNR／SSIM。**Preset 是「編碼時間 vs 壓縮效率」的取捨**：同樣 CRF 下，較慢的 preset 會用較少位元達到相近品質，或用相近位元達到較高品質。

## 13.22–13.24 ffprobe 與 Video Technical Card

**`ffmpeg` 負責「做」（轉檔、剪輯、濾鏡），`ffprobe` 負責「看」（讀出容器與串流的技術資訊，不改任何東西）。** 拿到一支影片，先 ffprobe 再決定怎麼處理。

講義的三個指令：

```bash
ffprobe -v error -show_format -show_streams input.mp4
ffprobe -v error -show_format -show_streams -of json input.mp4
ffprobe -v error -show_entries stream=index,codec_type,codec_name,width,height,pix_fmt,r_frame_rate,avg_frame_rate,bit_rate,sample_rate,channels:format=format_name,duration,size,bit_rate -of json input.mp4
```

- `-v error`：只顯示錯誤（不印版本資訊）。
- `-show_format`：容器層（格式、時長、大小、總位元率、tags）；`-show_streams`：每條串流（codec、解析度、pix_fmt、fps、取樣率……）。
- `-of json`：輸出 JSON，方便程式解析；`-show_entries`：只挑需要的欄位。

> **先預測再執行**：`input.mp4` 有幾條串流？`format_name` 會寫「mp4」嗎？

In [ ]:
FFPROBE_1 = "ffprobe -v error -show_format -show_streams input.mp4"
FFPROBE_2 = "ffprobe -v error -show_format -show_streams -of json input.mp4"
FFPROBE_3 = ("ffprobe -v error -show_entries stream=index,codec_type,codec_name,width,height,pix_fmt,r_frame_rate,"
             "avg_frame_rate,bit_rate,sample_rate,channels:format=format_name,duration,size,bit_rate -of json input.mp4")

p1, _ = run(FFPROBE_1, show=True)
out1 = p1.stdout.splitlines()
print("\n".join(out1[:18]), f"\n…（共 {len(out1)} 行）\n")
p2, _ = run(FFPROBE_2, show=True)
full = json.loads(p2.stdout)
print(f"JSON：streams × {len(full['streams'])}，format 欄位 {len(full['format'])} 個，tags = {full['format'].get('tags')}\n")
p3, _ = run(FFPROBE_3, show=True)
print(p3.stdout)

### 把 JSON 變成 Video Technical Card

講義 13.24 的範本：

```markdown
## Video Technical Card

Container: MP4
Video Codec: H.264 / AVC
Resolution: 1920 × 1080
Frame Rate: 30 fps
Pixel Format: yuv420p
Audio Codec: AAC
Audio Sample Rate: 48000 Hz
Channels: Stereo
Duration: 60.2 s
File Size: 61 MB
```

`technical_card()` 把 ffprobe 的 JSON 轉成同樣格式；缺少的欄位（例如沒有聲音、或值是 `N/A`）會寫 `N/A`。`extra=True` 時另外附上 Encoder（從 `tags` 讀取，若有）、影像與聲音位元率。先用一份「假的」ffprobe 結果確認輸出和範本**一字不差**。

In [ ]:
CARD_TEMPLATE = (
    "## Video Technical Card\n\nContainer: MP4\nVideo Codec: H.264 / AVC\nResolution: 1920 × 1080\n"
    "Frame Rate: 30 fps\nPixel Format: yuv420p\nAudio Codec: AAC\nAudio Sample Rate: 48000 Hz\n"
    "Channels: Stereo\nDuration: 60.2 s\nFile Size: 61 MB")
CODEC_NAMES = {"h264": "H.264 / AVC", "hevc": "H.265 / HEVC", "av1": "AV1", "vp9": "VP9", "vp8": "VP8",
               "mpeg4": "MPEG-4 Part 2", "prores": "Apple ProRes", "mjpeg": "Motion JPEG", "ffv1": "FFV1",
               "aac": "AAC", "mp3": "MP3", "opus": "Opus", "vorbis": "Vorbis", "ac3": "AC-3", "eac3": "E-AC-3",
               "flac": "FLAC", "alac": "ALAC", "pcm_s16le": "PCM 16-bit"}
NA = "N/A"


def _ok(x):
    return x not in (None, "", "N/A", "0/0")


def container_name(format_name, filename):
    fmt, ext = (format_name or ""), Path(filename).suffix.lower()
    if "mp4" in fmt or "mov" in fmt:
        return {".mov": "MOV", ".m4a": "M4A", ".3gp": "3GP"}.get(ext, "MP4")
    if "matroska" in fmt or "webm" in fmt:
        return "WebM" if ext == ".webm" else "MKV"
    return {"mpegts": "MPEG-TS", "avi": "AVI", "flv": "FLV", "ogg": "Ogg"}.get(fmt, fmt.upper() or NA)


def fmt_fps(s_avg, s_r):
    v = fps_value(s_avg) if _ok(s_avg) else None
    if v is None:
        return NA
    txt = f"{v:.0f} fps" if abs(v - round(v)) < 1e-6 else f"{v:.3f}".rstrip("0") + " fps"
    r = fps_value(s_r) if _ok(s_r) else None
    if r and abs(r - v) > 0.01 * r:
        txt += f"（可能是 VFR：r_frame_rate={s_r}）"
    return txt


def fmt_size(nbytes):
    mb = nbytes / 1e6
    return f"{mb:.0f} MB" if mb >= 10 else (f"{mb:.1f} MB" if mb >= 1 else f"{mb:.2f} MB")


def technical_card(probe_json, filename, extra=False, tags_json=None):
    streams = probe_json.get("streams", [])
    fmt = probe_json.get("format", {})
    v = next((s for s in streams if s.get("codec_type") == "video"), {})
    a = next((s for s in streams if s.get("codec_type") == "audio"), {})
    ch = a.get("channels")
    vals = {
        "Container": container_name(fmt.get("format_name"), filename),
        "Video Codec": CODEC_NAMES.get(v.get("codec_name"), v.get("codec_name") or NA),
        "Resolution": f"{v['width']} × {v['height']}" if _ok(v.get("width")) and _ok(v.get("height")) else NA,
        "Frame Rate": fmt_fps(v.get("avg_frame_rate"), v.get("r_frame_rate")),
        "Pixel Format": v.get("pix_fmt") or NA,
        "Audio Codec": CODEC_NAMES.get(a.get("codec_name"), a.get("codec_name") or NA),
        "Audio Sample Rate": f"{a['sample_rate']} Hz" if _ok(a.get("sample_rate")) else NA,
        "Channels": {1: "Mono", 2: "Stereo", 6: "5.1"}.get(ch, f"{ch} ch" if ch else NA),
        "Duration": f"{float(fmt['duration']):.1f} s" if _ok(fmt.get("duration")) else NA,
        "File Size": fmt_size(int(fmt["size"])) if _ok(fmt.get("size")) else NA,
    }
    lines = ["## Video Technical Card", ""] + [f"{k}: {val}" for k, val in vals.items()]
    if extra:
        t = tags_json or probe_json
        tv = next((s for s in t.get("streams", []) if s.get("codec_type") == "video"), {})
        enc = (tv.get("tags") or {}).get("encoder") or (t.get("format", {}).get("tags") or {}).get("encoder") or NA
        vb = to_float(v.get("bit_rate"))
        ab = to_float(a.get("bit_rate"))
        lines += ["", f"Encoder (tag): {enc}",
                  f"Video Bitrate: {vb / 1e6:.2f} Mbps" if vb else "Video Bitrate: N/A",
                  f"Audio Bitrate: {ab / 1e3:.0f} kbps" if ab else "Audio Bitrate: N/A"]
    return "\n".join(lines)


fake = {"streams": [{"codec_type": "video", "codec_name": "h264", "width": 1920, "height": 1080, "pix_fmt": "yuv420p",
                     "r_frame_rate": "30/1", "avg_frame_rate": "30/1"},
                    {"codec_type": "audio", "codec_name": "aac", "sample_rate": "48000", "channels": 2}],
        "format": {"format_name": "mov,mp4,m4a,3gp,3g2,mj2", "duration": "60.2", "size": "61000000"}}
assert technical_card(fake, "example.mp4") == CARD_TEMPLATE, "與講義範本不一致"
print("technical_card() 對範本資料的輸出與講義完全相同\n")

card = technical_card(json.loads(p3.stdout), "input.mp4", extra=True, tags_json=full)
Path("input_card.md").write_text(card, encoding="utf-8")
display(Markdown(card))

In [ ]:
# 其他檔案的卡片：沒有聲音、不是 H.264、改了副檔名、VFR
entries = ("stream=index,codec_type,codec_name,width,height,pix_fmt,r_frame_rate,avg_frame_rate,bit_rate,"
           "sample_rate,channels:format=format_name,duration,size,bit_rate")
cards = []
for f in ["video_only.mp4", "renamed.mkv", "vfr.mp4"] + sorted(str(x) for x in Path(".").glob("not_h264_*.mp4"))[:1]:
    c = technical_card(probe(f, entries), f).replace("## Video Technical Card", f"## Video Technical Card：`{f}`")
    cards.append(c)
display(Markdown("\n\n---\n\n".join(cards)))

**觀察**

- 第一個指令輸出很長（每條串流幾十個欄位）；第三個只挑需要的欄位，適合寫進報告或程式。
- `format_name` 是 `mov,mp4,m4a,3gp,3g2,mj2`——這是 FFmpeg「MOV／MP4 家族」解析器的名字，所以卡片上的 Container 還要搭配副檔名與 `major_brand` 判斷。
- `renamed.mkv` 的卡片仍寫 MP4；`video_only.mp4` 的聲音欄位是 N/A；`vfr.mp4` 的 Frame Rate 會被標註「可能是 VFR」。
- GUI 工具 **MediaInfo**（BSD-2-Clause）可以一鍵顯示這些資訊，適合沒有命令列的電腦；但寫報告時，ffprobe 的指令＋輸出更容易被別人**重現**。

## 13.25 控制實驗：一次只改一個變因

**概念**：「哪個設定比較好」要靠實驗回答。科學的控制實驗**一次只改一個變因**，其他全部固定，並且記錄：指令、ffprobe 結果、檔案大小、編碼時間、品質指標與截圖。

| 實驗 | 自變因（只改這個）| 固定 | 依變因（量測）|
|---|---|---|---|
| A | CRF：18／23／28 | 來源、codec（libx264）、解析度、fps、preset medium、聲音 copy | 大小、位元率、時間、PSNR、SSIM、VMAF、截圖 |
| B | 解析度：1920×1080 vs 1280×720 | 來源、CRF 23、preset medium、fps | 同上（品質要放大回 1080p 才能和參考比）|
| C | 場景複雜度：靜態 vs 高動態 | CRF 23、preset、解析度、長度（各 3 s）| 大小、位元率、品質 |

### 品質指標（都是 full-reference：需要「原始參考影片」）

| 指標 | 範圍 | 意義 | 限制 |
|---|---|---|---|
| **PSNR** | dB，越高越好（> 40 很接近、< 30 明顯失真）| 像素誤差（MSE）的對數 | 和人眼感受不完全一致 |
| **SSIM** | 0–1，越接近 1 越好 | 比較局部亮度、對比、結構 | 對色彩、時間抖動不敏感 |
| **VMAF** | 0–100，越高越好 | Netflix 開發的**感知品質**模型：結合多種特徵，用人類評分訓練 | 依觀看條件訓練（預設模型 `vmaf_v0.6.1` 假設在 1080p 電視上觀看）；不同解析度要先放大到同尺寸；對某些處理（例如銳化）可能高估；它是統計模型，不是「真理」|

**工具：Netflix VMAF**（[libvmaf](https://github.com/Netflix/vmaf)，BSD-2-Clause-Patent）透過 FFmpeg 的 `libvmaf` 濾鏡使用。Colab 內建的 FFmpeg 通常**沒有**編入 libvmaf，所以下一格會：(1) 先檢查 `ffmpeg -filters`；(2) 沒有的話，嘗試下載**含 libvmaf 的靜態版 FFmpeg**（只用來算 VMAF，其他步驟仍用系統的 FFmpeg）；(3) 下載失敗就只用 PSNR／SSIM，並清楚標示。

FFmpeg 內建的 `psnr`、`ssim` 濾鏡用法（第一個輸入是壓縮後、第二個是參考）。`quality()`／`vmaf()` 實際執行時會先用 `settb`＋`setpts=N` 把兩邊依影格順序對齊（見 13.18 的小插曲），其餘相同：

```bash
ffmpeg -i crf23.mp4 -i input.mp4 -lavfi "[0:v][1:v]psnr" -f null -
ffmpeg -i crf23.mp4 -i input.mp4 -lavfi "[0:v][1:v]ssim" -f null -
ffmpeg -i crf23.mp4 -i input.mp4 -lavfi "[0:v][1:v]libvmaf=log_fmt=json:log_path=vmaf_crf23.json" -f null -
```

> 嚴格來說，我們的參考 `input.mp4` 本身已是 CRF 18 壓縮過的「母帶」；真實研究會用無損原始檔。這不影響「比較 CRF 18／23／28」的結論，但 CRF 18 的分數會偏高（它和母帶設定接近）。

In [ ]:
TRY_STATIC_FFMPEG = True     # 系統 FFmpeg 沒有 libvmaf 時，是否嘗試下載靜態版（約 80–160 MB）
STATIC_URLS = [
    "https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz",
    "https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz",
]
VMAF_FFMPEG = "ffmpeg" if "libvmaf" in FILTERS else None
if VMAF_FFMPEG:
    print("系統 FFmpeg 已內建 libvmaf")
elif not TRY_STATIC_FFMPEG:
    print("TRY_STATIC_FFMPEG = False：只計算 PSNR／SSIM")
elif not (sys.platform.startswith("linux") and platform.machine() == "x86_64" and shutil.which("curl")):
    print("非 Linux x86_64 或沒有 curl：無法使用靜態版，只計算 PSNR／SSIM")
else:
    for i, url in enumerate(STATIC_URLS):
        dest = Path(f"ffmpeg_static/{i}")
        dest.mkdir(parents=True, exist_ok=True)
        tarball = dest / "ffmpeg.tar.xz"
        try:
            print(f"下載 {url.split('/')[2]} 的靜態版 FFmpeg…")
            subprocess.run(["curl", "-fsSL", "--max-time", "240", "-o", str(tarball), url], check=True, timeout=300)
            subprocess.run(["tar", "-xf", str(tarball), "-C", str(dest)], check=True, timeout=180)
            tarball.unlink()
            binary = next(b for b in dest.rglob("ffmpeg") if b.is_file() and os.access(b, os.X_OK))
            if "libvmaf" in ffmpeg_names("filters", binary=str(binary)):
                VMAF_FFMPEG = str(binary)
                print("可用：", VMAF_FFMPEG)
                break
            print("這個靜態版沒有 libvmaf")
        except Exception as err:
            print(f"  失敗（{type(err).__name__}）：{str(err)[:120]}")
STATUS["VMAF（libvmaf）"] = "real" if VMAF_FFMPEG else "fallback"
if not VMAF_FFMPEG:
    print("→ 本次只用 PSNR／SSIM（FFmpeg 內建濾鏡）。報告中請寫明沒有 VMAF。")


def vmaf(dist, ref, log_path, scale_to=None, ref_opts=(), subsample=5):
    # 回傳 VMAF 平均分數（每 subsample 張取 1 張計算以節省時間）；不可用時回傳 None
    if not VMAF_FFMPEG:
        return None
    for opts in (f":n_subsample={subsample}:n_threads=2", ""):
        graph = f"{align_graph(scale_to)}[d][r]libvmaf=log_fmt=json:log_path={log_path}{opts}"
        p, _ = run([VMAF_FFMPEG, "-hide_banner", "-nostats", "-i", str(dist), *ref_opts, "-i", str(ref),
                    "-lavfi", graph, "-f", "null", "-"], quiet=False, check=False)
        if p.returncode == 0:
            break
    try:
        j = json.loads(Path(log_path).read_text())
        if "pooled_metrics" in j:                                   # libvmaf 2.x
            return round(j["pooled_metrics"]["vmaf"]["mean"], 2)
        return round(j.get("VMAF score") or j["aggregate"]["VMAF_score"], 2)   # libvmaf 1.x
    except Exception:
        hit = re.findall(r"VMAF score[:=]\s*([\d.]+)", p.stderr)
        return float(hit[-1]) if hit else None

### 實驗 A：只改 CRF（18／23／28）

講義指令原樣執行（每支約 15 秒）：

In [ ]:
EXP_A = [
    "ffmpeg -i input.mp4 -c:v libx264 -crf 18 -preset medium -c:a copy crf18.mp4",
    "ffmpeg -i input.mp4 -c:v libx264 -crf 23 -preset medium -c:a copy crf23.mp4",
    "ffmpeg -i input.mp4 -c:v libx264 -crf 28 -preset medium -c:a copy crf28.mp4",
]


def measure(out, sec, ref="input.mp4", scale_to=None, ref_opts=(), label=None, psnr_log=None):
    d = info(out)
    q = quality(out, ref, scale_to=scale_to, ref_opts=ref_opts, psnr_log=psnr_log)
    v = vmaf(out, ref, Path(out).with_suffix(".vmaf.json"), scale_to=scale_to, ref_opts=ref_opts)
    return dict(version=label or Path(out).stem, file=out, resolution=f"{d['width']}x{d['height']}",
                size_MB=round(d["size_MB"], 2), bitrate_Mbps=round(d["bitrate_Mbps"], 2), encode_s=round(sec, 1),
                PSNR_dB=q["PSNR_dB"], SSIM=q["SSIM"], VMAF=v)


rows_a = []
for cmd in EXP_A:
    out = cmd.split()[-1]
    _, sec = run(cmd, show=True)
    rows_a.append(measure(out, sec, label=f"CRF {cmd.split()[6]}", psnr_log=f"{Path(out).stem}_psnr.log"))
EXP_A_RESULTS = pd.DataFrame(rows_a)
display(EXP_A_RESULTS)

In [ ]:
crfs = [18, 23, 28]
ncols = 3 if EXP_A_RESULTS.VMAF.notna().any() else 2
fig, axes = plt.subplots(1, ncols, figsize=(4.2 * ncols, 3.2))
axes[0].plot(crfs, EXP_A_RESULTS.size_MB, "o-")
axes[0].set_xlabel("CRF")
axes[0].set_ylabel("File size (MB)")
axes[0].set_title("Size vs CRF")
axes[1].plot(crfs, EXP_A_RESULTS.PSNR_dB, "o-", color="#c44e52", label="PSNR (dB)")
axes[1].set_xlabel("CRF")
axes[1].set_ylabel("PSNR (dB)")
ax2 = axes[1].twinx()
ax2.plot(crfs, EXP_A_RESULTS.SSIM, "s--", color="#55a868", label="SSIM")
ax2.set_ylabel("SSIM")
axes[1].set_title("Quality vs CRF (PSNR solid, SSIM dashed)")
if ncols == 3:
    axes[2].plot(crfs, EXP_A_RESULTS.VMAF, "o-", color="#8172b2")
    axes[2].set_xlabel("CRF")
    axes[2].set_ylabel("VMAF")
    axes[2].set_title("VMAF vs CRF")
plt.tight_layout()
plt.show()


def read_psnr_log(path):
    vals = [float(m) if m != "inf" else 99.0 for m in re.findall(r"psnr_avg:([\d.]+|inf)", Path(path).read_text())]
    return np.array(vals)


fig, ax = plt.subplots(figsize=(11, 3))
for crf in crfs:
    v = read_psnr_log(f"crf{crf}_psnr.log")
    ax.plot(np.arange(len(v)) / FPS, v, label=f"CRF {crf}")
for _, t0, _ in SCENES[1:]:
    ax.axvline(t0, color="k", ls="--", lw=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("PSNR per frame (dB)")
ax.set_title("Per-frame PSNR: same CRF, different quality per scene")
ax.legend()
plt.show()

for name, t, (y0, x0), hgt, wid in [("text", 1.0, (392, 150), 96, 240), ("motion", 5.0, (912, 888), 96, 240)]:
    crops = [grab(f, t)[y0:y0 + hgt, x0:x0 + wid] for f in ["input.mp4", "crf18.mp4", "crf23.mp4", "crf28.mp4"]]
    save_crops(crops, ["reference", "CRF 18", "CRF 23", "CRF 28"], f"crf_crops_{name}.png", zoom=2,
               title=f"Experiment A, {name} region at t={t}s: 1:1 crops")

**觀察（實驗 A）**

- CRF 每增加 5，檔案大約縮小為原本的 5–6 成；PSNR／SSIM（與 VMAF）一起下降。**CRF 是「品質等級」，不是位元率**。
- 逐格 PSNR 圖：同一個 CRF 下，靜態投影片的 PSNR 很高、高動態場景明顯較低——CRF 的「固定品質」是針對**感知**做的調整（x264 會在快速運動處多給一點失真，因為人眼較不易察覺），不是固定 PSNR。
- 截圖：靜態文字區三個版本幾乎看不出差別（靜態畫面很便宜，編碼器給得起位元）；高動態區的細碎雜訊紋理與棋盤格邊緣，隨 CRF 增加先被抹平。
- VMAF（若有）三個版本都很高、差距比 PSNR 小：VMAF 模擬的是「正常觀看距離下」人眼的感受，像素級的小誤差不一定看得出來。

### 實驗 B：只改解析度（1920×1080 vs 1280×720）

1080p 版本直接沿用實驗 A 的 `crf23.mp4`（設定完全相同：libx264、CRF 23、preset medium、聲音 copy）；720p 版本只多一個 `-vf scale=1280:720`。品質量測時，先把 720p **放大回 1920×1080** 再和參考比較（觀眾在 1080p 螢幕上看到的就是放大後的畫面）。

In [ ]:
EXP_B_720 = "ffmpeg -i input.mp4 -vf scale=1280:720 -c:v libx264 -crf 23 -preset medium -c:a copy res720.mp4"
_, sec720 = run(EXP_B_720, show=True)
rows_b = [dict(rows_a[1], version="1920x1080 (crf23.mp4)"),
          measure("res720.mp4", sec720, scale_to=(1920, 1080), label="1280x720 (upscaled for metrics)")]
EXP_B_RESULTS = pd.DataFrame(rows_b)
display(EXP_B_RESULTS)
crops = [grab(f, 1.0, size=(1920, 1080))[392:488, 150:390] for f in ["input.mp4", "crf23.mp4", "res720.mp4"]]
save_crops(crops, ["reference", "1080p CRF 23", "720p CRF 23 (upscaled)"], "res_crops.png", zoom=2,
           title="Experiment B: 1:1 crops at 1080p display size")
print(f"像素數比 1080p / 720p = {pixels(1920, 1080) / pixels(1280, 720):.2f}；檔案大小比 = "
      f"{rows_b[0]['size_MB'] / rows_b[1]['size_MB']:.2f}")

### 實驗 C：只改場景複雜度（靜態 vs 高動態）

從 `input.mp4` 各取 3 秒：0–3 s（靜態投影片）與 4–7 s（高動態），用**完全相同**的設定編碼（不含聲音，避免聲音成為另一個變因）。

In [ ]:
EXP_C = {
    "static (0-3 s)": ("0", "ffmpeg -ss 0 -i input.mp4 -t 3 -an -c:v libx264 -crf 23 -preset medium scene_static.mp4"),
    "high-motion (4-7 s)": ("4", "ffmpeg -ss 4 -i input.mp4 -t 3 -an -c:v libx264 -crf 23 -preset medium scene_motion.mp4"),
}
rows_c = []
for label, (ss, cmd) in EXP_C.items():
    _, sec = run(cmd, show=True)
    rows_c.append(measure(cmd.split()[-1], sec, ref_opts=("-ss", ss, "-t", "3"), label=label))
EXP_C_RESULTS = pd.DataFrame(rows_c)
display(EXP_C_RESULTS)
print(f"同樣 CRF 23、同樣 3 秒：高動態 / 靜態 的檔案大小 = {rows_c[1]['size_MB'] / rows_c[0]['size_MB']:.1f} 倍")

**觀察（實驗 B、C）**

- 實驗 B：720p 的像素只有 1080p 的 44%，檔案通常也小很多；但放大回 1080p 後，文字邊緣明顯變軟、PSNR／VMAF 下降。**4K（4 倍像素）也不會剛好 4 倍大**——編碼器在高解析度下能利用更多空間冗餘。
- 實驗 C：設定完全相同，只因為內容不同，檔案大小差了好幾十倍、品質也不同。**位元率 ≠ 品質**；比較不同影片的位元率時，一定要說明內容。
- 注意指標之間可能**不一致**：PSNR 說靜態場景好很多，VMAF（若有）卻可能給高動態場景差不多甚至更高的分數——VMAF 考慮了「動作會遮蔽瑕疵」（temporal masking）。報告時至少列兩種指標，並附截圖。
- 三個實驗都是「一次只改一個變因」：如果同時改 CRF 和解析度，就無法說清楚檔案變小是哪一個造成的。

## 開源工具總整理（課後可以安裝）

| 工具 | 授權 | 用途 | 本週對應 |
|---|---|---|---|
| [FFmpeg](https://ffmpeg.org/)／ffprobe | LGPL-2.1+（含 x264 等時為 GPL）| 命令列轉檔、剪輯、量測 | 全部 |
| [PyAV](https://github.com/PyAV-Org/PyAV) | BSD-3-Clause | 在 Python 中逐 packet／frame 存取 | 13.10、13.12 |
| [Netflix VMAF](https://github.com/Netflix/vmaf) | BSD-2-Clause-Patent | 感知品質指標（FFmpeg `libvmaf` 濾鏡）| 13.25 |
| [PySceneDetect](https://www.scenedetect.com/) | BSD-3-Clause | 場景切換偵測（也有命令列 `scenedetect`）| 13.14 |
| [LosslessCut](https://github.com/mifi/lossless-cut) | GPL-2.0 | **無損剪輯**的 GUI（底層就是 `-c copy`）；極快，但起點會對齊到前一個關鍵影格 | 13.15 |
| [MediaInfo](https://mediaarea.net/MediaInfo) | BSD-2-Clause | 一鍵看技術資訊（GUI 版的 Technical Card）| 13.22–13.24 |
| [HandBrake](https://handbrake.fr/) | GPL-2.0 | GUI 轉檔器：Presets、RF（= x264／x265 的 CRF）、H.264／H.265／AV1 | 13.18–13.21 |

**瀏覽器版互動實驗室**（[W13 Lab](https://willismax.github.io/MediaSystem-Python-Course/labs/w13/)）使用瀏覽器內建的 **WebCodecs API**：JavaScript 可以直接呼叫瀏覽器（甚至硬體）的編碼器、解碼器，逐張影格操作——概念和 PyAV 一樣，只是在瀏覽器裡。不同瀏覽器支援的 codec 不同，正好呼應 13.18 的「解碼相容性」。

## 練習 1：自己影片的 Video Technical Card

1. 在 0.5 上傳自己用手機拍的影片（或設定 `MY_VIDEO_PATH`）。
2. 執行下一格：用講義 13.23 的 ffprobe 指令產生 Technical Card，存成 `ex1_card.md`。
3. 填寫下表，特別注意：是不是 VFR？是不是 HEVC？像素格式是不是 yuv420p？（iPhone 常是 HEVC、HDR 影片可能是 10-bit 的 `yuv420p10le`）

| 項目 | 內容 |
|---|---|
| 拍攝裝置／App | |
| 錄影設定（解析度、fps、HDR？）| |
| `r_frame_rate` vs `avg_frame_rate` | |
| 和 `input.mp4` 最不一樣的欄位 | |
| 如果要交給「只支援 H.264」的平台，要改哪些設定？| |

In [ ]:
EX1_VIDEO = MY_VIDEO or Path("input.mp4")            # TODO：沒有上傳時請改成自己的影片
if MY_VIDEO is None:
    print("（示範：目前用 input.mp4；繳交時請換成自己的影片）")
ex1_cmd = FFPROBE_3.replace("input.mp4", shlex.quote(str(EX1_VIDEO)))
p, _ = run(ex1_cmd, show=True)
ex1_full = probe(EX1_VIDEO)
ex1_card = technical_card(json.loads(p.stdout), EX1_VIDEO, extra=True, tags_json=ex1_full)
Path("ex1_card.md").write_text(ex1_card, encoding="utf-8")
display(Markdown(ex1_card))
vs = next((s for s in ex1_full.get("streams", []) if s.get("codec_type") == "video"), {})
print("其他值得注意的欄位：", {k: vs.get(k) for k in ["profile", "level", "color_space", "color_transfer",
                                                    "color_primaries", "bits_per_raw_sample"] if vs.get(k)})
print("format tags：", (ex1_full.get("format", {}).get("tags") or {}))

## 練習 2：未壓縮資料量 vs 手機錄影

計算 60 秒、30 fps、RGB 8-bit 的未壓縮資料量，並和真實錄影比較：

| 規格 | 計算式 | 未壓縮 | 同長度實際錄影（手機）| 壓縮比 |
|---|---|---|---|---|
| 1280×720／30 fps／60 s | 1280 × 720 × 3 × 30 × 60 | | | |
| 1920×1080／30 fps／60 s | | | | |
| 3840×2160／30 fps／60 s | | | | |

In [ ]:
def raw_bytes(w, h, fps, seconds, bpp=3):
    return w * h * bpp * fps * seconds


assert raw_bytes(1280, 720, 30, 60) == 4_976_640_000            # ≈ 4.98 GB
assert raw_bytes(3840, 2160, 30, 60) == 44_789_760_000          # ≈ 44.79 GB
assert round(raw_bytes(3840, 2160, 30, 60) / 1e9, 2) == 44.79

PHONE_VIDEO = EX1_VIDEO                                          # TODO：你的手機影片
ph = info(PHONE_VIDEO)
phone_60s_MB = file_size_MB(ph["bitrate_Mbps"], 60)
rows = []
for w, h in [(1280, 720), (1920, 1080), (3840, 2160)]:
    raw = raw_bytes(w, h, 30, 60)
    same = (ph["width"], ph["height"]) == (w, h)
    rows.append(dict(spec=f"{w}x{h} / 30 fps / 60 s", formula=f"{w} × {h} × 3 × 30 × 60",
                     raw_bytes=f"{raw:,}", raw_GB=round(raw / 1e9, 2),
                     recorded_60s_MB=round(phone_60s_MB, 1) if same else None,
                     ratio=f"{raw / (phone_60s_MB * 1e6):.0f} : 1" if same else None))
display(pd.DataFrame(rows))
print(f"{PHONE_VIDEO}：{ph['width']}×{ph['height']}、{ph['fps']:.2f} fps、{ph['bitrate_Mbps']:.2f} Mbps → "
      f"60 秒約 {phone_60s_MB:.1f} MB（只有相同解析度那一列會填入實際值）")

## 練習 3：CRF 控制實驗（至少 3 個版本）

1. 選一段自己的影片（10 秒左右即可；超過會自動只取前 `EX3_SECONDS` 秒）。
2. 只改 CRF（至少 3 個值），其他全部固定；執行後會得到表格、ffprobe 資訊、截圖 `ex3_crops.png`。
3. 寫結論：你會選哪個 CRF？理由要引用**大小、品質數字、截圖**，並說明這個結論只適用於什麼內容。

| 版本 | CRF | 檔案大小 | 位元率 | 編碼時間 | PSNR／SSIM／VMAF | 截圖觀察 |
|---|---|---|---|---|---|---|
| 1 | | | | | | |
| 2 | | | | | | |
| 3 | | | | | | |

結論（只談 CRF 這個變因）：

In [ ]:
EX3_SOURCE = MY_VIDEO or Path("input.mp4")       # TODO：你的影片
EX3_CRFS = [18, 23, 28]                          # TODO：至少 3 個值
EX3_SECONDS = 10
EX3_SHOT_T = 1.0                                 # TODO：截圖時間點（秒）

if EX3_SOURCE == Path("input.mp4") and EX3_CRFS == [18, 23, 28]:
    print("（示範：來源與 CRF 和實驗 A 相同，直接沿用實驗 A 的結果；換成自己的影片後會重新編碼）")
    ex3_df = EXP_A_RESULTS.copy()
    ex3_files = ["crf18.mp4", "crf23.mp4", "crf28.mp4"]
    ex3_ref = "input.mp4"
else:
    ex3_ref = "ex3_source.mp4"
    run(["ffmpeg", "-i", str(EX3_SOURCE), "-t", str(EX3_SECONDS), "-c", "copy", ex3_ref])
    ex3_rows, ex3_files = [], []
    for crf in EX3_CRFS:
        out = f"ex3_crf{crf}.mp4"
        _, sec = run(["ffmpeg", "-i", ex3_ref, "-c:v", "libx264", "-crf", str(crf), "-preset", "medium",
                      "-c:a", "copy", out], show=True)
        ex3_rows.append(measure(out, sec, ref=ex3_ref, label=f"CRF {crf}"))
        ex3_files.append(out)
    ex3_df = pd.DataFrame(ex3_rows)
display(ex3_df)
w3, h3 = video_size(ex3_ref)
cy, cx = h3 // 3, w3 // 3                         # TODO：改成畫面中有細節的位置
crops = [grab(f, EX3_SHOT_T)[cy:cy + 160, cx:cx + 240] for f in [ex3_ref, *ex3_files]]
save_crops(crops, ["source", *[f"CRF {c}" for c in EX3_CRFS]], "ex3_crops.png", zoom=2,
           title="Exercise 3: 1:1 crops")

## 延伸 1：靜態場景 vs 動態場景

拍兩段**同樣長度、同樣設定**的影片：一段幾乎不動（拍桌上的書）、一段大幅運動（走路時拍、拍車流），各 10 秒，填入 `STATIC_CLIP`、`DYNAMIC_CLIP`。沒有提供時，下面顯示實驗 C 的結果。

| 場景 | 原始位元率 | CRF 23 後大小 | 品質 | 你的觀察 |
|---|---|---|---|---|
| 靜態 | | | | |
| 動態 | | | | |

In [ ]:
STATIC_CLIP, DYNAMIC_CLIP = None, None           # TODO：例如 "static.mp4", "dynamic.mp4"
if STATIC_CLIP and DYNAMIC_CLIP:
    rows = []
    for lab, src in [("static", STATIC_CLIP), ("dynamic", DYNAMIC_CLIP)]:
        out = f"ext1_{lab}.mp4"
        _, sec = run(["ffmpeg", "-i", src, "-t", "10", "-an", "-c:v", "libx264", "-crf", "23", "-preset", "medium", out])
        rows.append(dict(measure(out, sec, ref=src, ref_opts=("-t", "10"), label=lab),
                         source_Mbps=round(info(src)["bitrate_Mbps"], 2)))
    display(pd.DataFrame(rows))
else:
    print("尚未提供自己的兩段影片；以下是實驗 C 的結果：")
    display(EXP_C_RESULTS)

## 延伸 2：Stream Copy vs 重新編碼剪輯

用自己的影片剪一段（起點刻意**不要**選在關鍵影格），比較兩種方法。`cut_compare()` 會印出來源的關鍵影格時間，先看再決定起點。

| 方法 | 執行時間 | 檔案大小 | 實際長度 | 第一個 packet 時間 | 你看到的第一張畫面 |
|---|---|---|---|---|---|
| Stream Copy | | | | | |
| 重新編碼 | | | | | |

In [ ]:
EXT2_SOURCE = MY_VIDEO or Path("input.mp4")      # TODO
EXT2_START, EXT2_DUR = 5, 4                       # TODO：依你的影片長度調整
if EXT2_SOURCE == Path("input.mp4"):
    print("（示範：沿用 13.15 的結果）")
    display(cut_df)
else:
    src = "ext2_source" + Path(EXT2_SOURCE).suffix
    shutil.copy(EXT2_SOURCE, src)                 # 複製成沒有空白的檔名，方便組指令
    display(cut_compare(src, EXT2_START, EXT2_DUR, tag="ext2"))

## 延伸 3：Codec 比較表

下表的 File Size、Encode Time、PSNR 由 13.18 自動填入（640×360、4 秒）。**Decode Compatibility** 與 **Visual Quality** 請你自己填：前者查資料或實測（手機、Chrome、Safari、學校電腦能不能播？），後者用眼睛看（建議把 `codec_*.mp4` 下載後並排播放）。

> 提醒：表中每個 codec 的 CRF 不同，數字不能直接解讀成「誰的壓縮效率高多少」。

In [ ]:
ext3 = CODEC_RESULTS[CODEC_RESULTS.codec.isin(["H.264", "H.265", "AV1 (SVT-AV1)", "AV1 (libaom)"])].copy()
ext3 = ext3.rename(columns={"size_KB": "File Size (KB)", "encode_s": "Encode Time (s)", "PSNR_dB": "PSNR (dB)"})
ext3["Decode Compatibility"] = "TODO"
ext3["Visual Quality"] = "TODO"
display(ext3[["codec", "encoder", "settings", "File Size (KB)", "Encode Time (s)", "PSNR (dB)",
              "Decode Compatibility", "Visual Quality"]])
missing = {"H.264", "H.265", "AV1"} - {c.split()[0] for c in ext3.codec}
if missing:
    print("這台機器缺少：", missing, "→ 可用 HandBrake 或本機 FFmpeg 補做，並在報告中註明工具與版本。")

## 執行摘要

列出本次執行中，各步驟是**真的執行**（real）、**用了替代方案**（fallback）還是**略過**（skipped），以及 FFmpeg 版本。報告中引用的數字請以 real 的結果為準，並寫出 FFmpeg 版本（不同版本的 x264／x265 結果會略有不同）。

In [ ]:
display(pd.DataFrame(sorted(STATUS.items()), columns=["步驟", "狀態"]))
print(f"FFmpeg {FFMPEG_VERSION}｜VMAF 使用：{VMAF_FFMPEG or '無'}｜PyAV {pkg_version('av')}｜scenedetect {pkg_version('scenedetect')}")
print(f"real {sum(v == 'real' for v in STATUS.values())} 項｜fallback／skipped {sum(v != 'real' for v in STATUS.values())} 項")

## 檢核題

請用自己的話回答（每題 2–4 句），盡量引用本 notebook 的數字、圖或截圖。

1. 一支 2 分 30 秒、29.97 fps 的影片大約有幾張影格？`r_frame_rate` 和 `avg_frame_rate` 不同代表什麼？
2. 4K 的像素是 1080p 的 4 倍，為什麼同樣設定下檔案通常**不會**剛好大 4 倍？
3. 寫出 1080p30 未壓縮 ≈ 1.49 Gbps 的計算過程；`input.mp4` 的壓縮比大約多少？
4. 為什麼「位元率高 ≠ 品質好」？請用 13.14 或實驗 C 的數字說明。
5. 畫面內（intra）與畫面間（inter）壓縮各利用了哪一種冗餘？
6. 在預測 → 轉換 → 量化 → 熵編碼中，哪一步造成資訊損失？為什麼 DCT 本身不會？
7. `yuv420p` 是解析度、codec，還是像素格式？4:2:0 每像素幾 bytes？為什麼犧牲色度而不是亮度？
8. 什麼是 motion vector？什麼是殘差？為什麼動作補償後殘差能量會大幅下降？
9. B 影格是「前後兩張影格的平均」嗎？為什麼有 B 影格時解碼順序和顯示順序不同？
10. GOP 長短如何影響檔案大小、拖曳時間軸（隨機存取）與錯誤傳播？
11. 用 `-c copy` 剪輯為什麼無法精準從任意時間點開始？13.15 的 `clip_copy.mp4` 多出了什麼？
12. Codec、Encoder、Container 各是什麼？各舉一例。
13. 為什麼 `.mp4` 不代表一定是 H.264？把 `.mp4` 改名成 `.mkv` 會發生什麼事？
14. CRF 與 preset 分別控制什麼？`ultrafast` 與 `slow` 的取捨是什麼？
15. `ffprobe` 和 `ffmpeg` 的角色有什麼不同？為什麼處理影片前要先 ffprobe？
16. 為什麼控制實驗一次只能改一個變因？請設計一個你想做的影片編碼實驗（自變因、固定條件、量測）。

### 我的答案

1.
2.
3.
4.
5.
6.
7.
8.
9.
10.
11.
12.
13.
14.
15.
16.

## 繳交清單

- [ ] 本 notebook（`.ipynb`），已執行完成並保留輸出；「執行摘要」中報告引用的步驟都是 real，並寫明 FFmpeg 版本
- [ ] 練習 1：自己影片的 Video Technical Card `ex1_card.md`＋觀察表（VFR？HEVC？pix_fmt？）
- [ ] 練習 2：720p／1080p／4K 未壓縮資料量計算表，與自己手機錄影的比較
- [ ] 練習 3：CRF 控制實驗（**至少 3 個版本**）：指令、ffprobe 結果、檔案大小、編碼時間、品質指標、截圖 `ex3_crops.png`、結論
- [ ] （延伸）靜態 vs 動態場景比較表、Stream Copy vs 重新編碼比較表、Codec 比較表（含 Decode Compatibility 與 Visual Quality）
- [ ] 檢核題 16 題作答

> 繳交前檢查自己影片的中繼資料（拍攝地點、裝置），不想公開就只交 Technical Card 與截圖。下載檔案：在 Colab 左側「檔案」面板對檔案按右鍵 → 下載。